# PlantDoc — Archive & Tools

**Do not run these cells in the same kernel as `PlantDoc_Main.ipynb`.** Several redefine functions the main notebook uses (e.g. the TTA cell redefines `_make_preprocess_fn`), which would silently change training.

Most cells were written for the v1 notebook and depend on its definitions — keep your original `PlantDoc_Pipeline_Local.ipynb` as the v1 record if you need to re-run them. Each cell still has its own settings block at the top.

## A. Data preparation & audits (already done — kept for reproducibility)

Superseded for the main workflow by `prepare_final_dataset.py` + `verify_clahe_dataset.py`.

#### def apply_clahe_offline():  
*(original cell 13)*

In [ ]:
import cv2
def apply_clahe_offline():
    if not CONFIG["use_clahe"]:
        return

    for split, src_dir in [("train", CONFIG["dataset_dir_train_src"]),
                            ("test",  CONFIG["dataset_dir_test_src"])]:
        for class_name in os.listdir(src_dir):
            src_class = os.path.join(src_dir, class_name)
            if not os.path.isdir(src_class):
                continue
            dst_class = os.path.join(CONFIG["clahe_output_dir"], split, class_name)
            os.makedirs(dst_class, exist_ok=True)

            for fname in os.listdir(src_class):
                dst_path = os.path.join(dst_class, fname)
                if os.path.exists(dst_path):   # skip if already processed
                    continue
                src_path = os.path.join(src_class, fname)
                img = cv2.imread(src_path)
                if img is None:
                    continue
                img = cv2.resize(img, (CONFIG["img_size"], CONFIG["img_size"]))
                lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                clahe = cv2.createCLAHE(
                    clipLimit=CONFIG["clahe_clip_limit"],
                    tileGridSize=CONFIG["clahe_tile_grid"]
                )
                lab[:, :, 0] = clahe.apply(lab[:, :, 0])
                img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
                cv2.imwrite(dst_path, img)

    print(f"[CLAHE] Done. Saved to {CONFIG['clahe_output_dir']}")


    

# apply_clahe_offline()

#### VERIFY CLAHE ON THE FINAL DATASET — every image, every split  
*(original cell 46)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY CLAHE ON THE FINAL DATASET — every image, every split
# For each image in dataset/{train,val,test}, re-runs your exact pipeline
# (resize -> LAB -> CLAHE on L -> BGR -> save) on the ORIGINAL file and compares.
#   match (diff ~ 0)   : exactly one CLAHE pass, correct size        -> OK
#   large diff         : e.g. CLAHE applied twice, or not at all      -> FLAGGED
#   wrong size         : raw / unprocessed image                      -> FLAGGED
#   no original found  : file not traceable to the original dataset   -> FLAGGED
# ═══════════════════════════════════════════════════════════════════════════════
import os
import cv2
import numpy as np

VERIFY = {
    "final_dataset":  "/home/auc/PlantDoc/Datasets/dataset",
    "original_train": "/home/auc/PlantDoc/Datasets/original/PlantDoc-Dataset/train",
    "original_test":  "/home/auc/PlantDoc/Datasets/original/PlantDoc-Dataset/test",
    "img_size": 224,
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid": (8, 8),
    "max_mean_diff": 0.5,      # 0-255 scale; one exact CLAHE pass gives ~0.0
}


def clahe_replica(path, cfg):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (cfg["img_size"], cfg["img_size"]))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=cfg["clahe_clip_limit"], tileGridSize=cfg["clahe_tile_grid"])
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    ext = os.path.splitext(path)[1].lower()
    ok, buf = cv2.imencode(ext if ext in (".jpg", ".jpeg", ".png", ".bmp", ".webp") else ".jpg", img)
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)   # same encode/decode round-trip as the saved files


def verify(cfg):
    flagged, summary = [], {}
    for split, orig_root in [("train", cfg["original_train"]), ("val", cfg["original_train"]),
                             ("test", cfg["original_test"])]:
        root = os.path.join(cfg["final_dataset"], split)
        n, diffs = 0, []
        for cls in sorted(os.listdir(root)):
            for f in sorted(os.listdir(os.path.join(root, cls))):
                if f.startswith("."):
                    continue
                n += 1
                rel = f"{cls}/{f}"
                final = cv2.imread(os.path.join(root, cls, f))
                orig = os.path.join(orig_root, cls, f)
                if final is None:
                    flagged.append((split, rel, "unreadable"))
                    continue
                if final.shape[:2] != (cfg["img_size"], cfg["img_size"]):
                    flagged.append((split, rel, f"size {final.shape[1]}x{final.shape[0]} (raw/unprocessed?)"))
                    continue
                if not os.path.exists(orig):
                    flagged.append((split, rel, "no original with this class/filename"))
                    continue
                rep = clahe_replica(orig, cfg)
                d = float(np.abs(rep.astype(np.int16) - final.astype(np.int16)).mean())
                diffs.append(d)
                if d > cfg["max_mean_diff"]:
                    flagged.append((split, rel, f"mean diff {d:.2f} (CLAHE twice, or different processing)"))
        summary[split] = (n, max(diffs) if diffs else 0.0)
        print(f"{split:5s}: {n} images checked | worst diff among matches {summary[split][1]:.4f}")

    print(f"\nFlagged: {len(flagged)}")
    for s, rel, why in flagged[:30]:
        print(f"  {s}: {rel} -> {why}")
    if len(flagged) > 30:
        print(f"  ... and {len(flagged) - 30} more")
    print("\nRESULT: " + ("every image is exactly one CLAHE pass of its original — OK"
                          if not flagged else "FIX THE FLAGGED IMAGES BEFORE TRAINING"))
    return flagged


flagged = verify(VERIFY)

#### DATA LEAKAGE / NEAR-DUPLICATE CHECK  
*(original cell 43)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA LEAKAGE / NEAR-DUPLICATE CHECK
# Detects near-identical images spanning train/val/test splits using perceptual
# hashing (robust to resizing, mild recompression, and minor crop differences —
# unlike exact byte/MD5 matching, which would miss most real-world duplicates).
#
# Run this separately per dataset (PlantVillage, PlantDoc, cassava, field images)
# by pointing TRAIN_DIR / TEST_DIR / VAL_DIR at that dataset's folders.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import imagehash
from PIL import Image
import pandas as pd
from itertools import combinations

# ── Configure here ────────────────────────────────────────────────────────────
DATASET_NAME = "Cassava"   # label for the report — change per dataset run

# Set MODE = "split" for datasets with fixed train/test (/val) folders on disk
# (PlantDoc, PlantVillage). Set MODE = "single_pool" for datasets that use
# k-fold cross-validation from ONE directory with no persistent train/test
# split on disk (e.g. Cassava) — this mode finds near-duplicates WITHIN the
# single pool, which you then cross-reference against your fold assignments
# to check whether any near-duplicate pair was split across two different
# folds (that pair would leak for whichever fold uses one as train and the
# other as validation).
MODE = "single_pool"   # "split" or "single_pool"

TRAIN_DIR = "./plantdoc_local/working/plantdoc_clahe/train"
TEST_DIR  = "./plantdoc_local/working/plantdoc_clahe/test"
VAL_DIR   = None   # set if you have a separate validation split; None to skip

SINGLE_POOL_DIR = "./plantdoc_local/input/competitions/cassava-leaf-disease-classification/train_images"   # only used when MODE == "single_pool"
FLAT_STRUCTURE = True   # set True if images sit directly in the folder with
                          # no class subfolders (e.g. Kaggle Cassava competition's
                          # train_images/ — labels come from a separate CSV, not
                          # folder names). Applies to whichever dir(s) MODE uses.

# Hamming distance threshold for flagging as a near-duplicate.
# 0 = exact/perceptually identical. Up to ~5 is still typically the "same
# photo" (recompressed, mildly resized, watermark added). Above ~10 is
# usually a genuinely different photo. Start conservative at 5.
HAMMING_THRESHOLD = 5

HASH_SIZE = 16  # larger = more precise but slower; 16 is a reasonable default

OUTPUT_REPORT = f"./plantdoc_local/working/leakage_check_{DATASET_NAME}.xlsx"


# ── Build hash index for one split ─────────────────────────────────────────────
def hash_split(root_dir, split_name, flat=False):
    """
    Walks root_dir and computes a perceptual hash for every image, returning
    a list of records.

    flat=False (default): expects root_dir/<class>/<image> structure
        (PlantDoc, PlantVillage style).
    flat=True: expects root_dir/<image> directly, no class subfolders
        (Kaggle Cassava competition style — train_images/ has files directly,
        with labels in a separate train.csv). "class" is recorded as "N/A"
        in this mode since it isn't available from the folder structure.
    """
    records = []
    if root_dir is None or not os.path.exists(root_dir):
        return records

    if flat:
        for fname in sorted(os.listdir(root_dir)):
            fpath = os.path.join(root_dir, fname)
            if not os.path.isfile(fpath):
                continue
            if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue
            try:
                with Image.open(fpath) as img:
                    phash = imagehash.phash(img, hash_size=HASH_SIZE)
            except Exception as e:
                print(f"  WARNING: could not hash {fpath}: {e}")
                continue
            records.append({
                "split": split_name,
                "class": "N/A",
                "filename": fname,
                "path": fpath,
                "hash": phash,
            })
    else:
        for cls in sorted(os.listdir(root_dir)):
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                try:
                    with Image.open(fpath) as img:
                        phash = imagehash.phash(img, hash_size=HASH_SIZE)
                except Exception as e:
                    print(f"  WARNING: could not hash {fpath}: {e}")
                    continue
                records.append({
                    "split": split_name,
                    "class": cls,
                    "filename": fname,
                    "path": fpath,
                    "hash": phash,
                })

    print(f"  Hashed {len(records)} images in {split_name} ({root_dir})")
    return records


# ── Compare two lists of records for near-duplicates ───────────────────────────
def find_near_duplicates(records_a, records_b, threshold, cross_split_only=True):
    """
    Compares every pair between records_a and records_b (or within one list,
    if records_a is records_b, for intra-split duplicate checking).
    cross_split_only: if True and comparing a list against itself, skips
    pairs from the same split (not meaningful — every image matches itself).
    Returns a list of flagged pairs with their hamming distance.
    """
    flagged = []
    same_list = records_a is records_b

    if same_list:
        pairs = combinations(range(len(records_a)), 2)
        for i, j in pairs:
            r1, r2 = records_a[i], records_a[j]
            dist = r1["hash"] - r2["hash"]
            if dist <= threshold:
                flagged.append({
                    "distance": dist,
                    "split_1": r1["split"], "class_1": r1["class"], "path_1": r1["path"],
                    "split_2": r2["split"], "class_2": r2["class"], "path_2": r2["path"],
                })
    else:
        for r1 in records_a:
            for r2 in records_b:
                dist = r1["hash"] - r2["hash"]
                if dist <= threshold:
                    flagged.append({
                        "distance": dist,
                        "split_1": r1["split"], "class_1": r1["class"], "path_1": r1["path"],
                        "split_2": r2["split"], "class_2": r2["class"], "path_2": r2["path"],
                    })
    return flagged


# ── Run the full check ──────────────────────────────────────────────────────────
print(f"Running data leakage check for: {DATASET_NAME}  (mode: {MODE})")
print(f"Hamming distance threshold: {HAMMING_THRESHOLD}\n")

if MODE == "single_pool":
    # ── K-fold dataset: find near-duplicates WITHIN the single pool ───────────
    pool_records = hash_split(SINGLE_POOL_DIR, "pool", flat=FLAT_STRUCTURE)

    print("\nChecking for near-duplicates within the pool...")
    within_pool_flagged = find_near_duplicates(pool_records, pool_records, HAMMING_THRESHOLD)
    print(f"  Found {len(within_pool_flagged)} near-duplicate pair(s) within the pool")

    with pd.ExcelWriter(OUTPUT_REPORT) as writer:
        if within_pool_flagged:
            pd.DataFrame(within_pool_flagged).sort_values("distance").to_excel(
                writer, sheet_name="Within_Pool_Duplicates", index=False
            )
        else:
            pd.DataFrame([{"result": "No near-duplicates found within the pool"}]).to_excel(
                writer, sheet_name="Within_Pool_Duplicates", index=False
            )
        summary = pd.DataFrame([{
            "dataset": DATASET_NAME, "mode": MODE,
            "pool_images": len(pool_records),
            "hamming_threshold": HAMMING_THRESHOLD,
            "within_pool_duplicates_flagged": len(within_pool_flagged),
        }])
        summary.to_excel(writer, sheet_name="Summary", index=False)

    print(f"\n{'='*60}")
    print(f"SUMMARY — {DATASET_NAME} (single_pool mode)")
    print(f"{'='*60}")
    print(f"  Pool images: {len(pool_records)}")
    print(f"  Within-pool near-duplicates flagged: {len(within_pool_flagged)}")
    print(f"\nFull report saved to: {OUTPUT_REPORT}")
    print(f"\nNEXT STEP: cross-reference the flagged pairs in the report against")
    print(f"your k-fold assignment file — for each flagged pair, check whether")
    print(f"the two images landed in the SAME fold (safe) or DIFFERENT folds")
    print(f"(that fold has a leak between its train and validation portions).")

else:
    # ── Fixed split dataset (PlantDoc, PlantVillage) ───────────────────────────
    train_records = hash_split(TRAIN_DIR, "train", flat=FLAT_STRUCTURE)
    test_records  = hash_split(TEST_DIR, "test", flat=FLAT_STRUCTURE)
    val_records   = hash_split(VAL_DIR, "val", flat=FLAT_STRUCTURE) if VAL_DIR else []

    all_flagged = []

    print("\nChecking train <-> test overlap...")
    flagged = find_near_duplicates(train_records, test_records, HAMMING_THRESHOLD)
    print(f"  Found {len(flagged)} potential leak(s)")
    all_flagged.extend(flagged)

    if val_records:
        print("Checking train <-> val overlap...")
        flagged = find_near_duplicates(train_records, val_records, HAMMING_THRESHOLD)
        print(f"  Found {len(flagged)} potential leak(s)")
        all_flagged.extend(flagged)

        print("Checking val <-> test overlap...")
        flagged = find_near_duplicates(val_records, test_records, HAMMING_THRESHOLD)
        print(f"  Found {len(flagged)} potential leak(s)")
        all_flagged.extend(flagged)

    # Also check WITHIN train for duplicate images across different classes
    # (a labeling error / mislabel risk, distinct from split leakage but worth flagging)
    print("Checking within-train cross-class duplicates (possible mislabels)...")
    within_train_flagged = []
    for i, j in combinations(range(len(train_records)), 2):
        r1, r2 = train_records[i], train_records[j]
        if r1["class"] == r2["class"]:
            continue  # same class duplicates within train are not a leakage concern
        dist = r1["hash"] - r2["hash"]
        if dist <= HAMMING_THRESHOLD:
            within_train_flagged.append({
                "distance": dist,
                "split_1": "train", "class_1": r1["class"], "path_1": r1["path"],
                "split_2": "train", "class_2": r2["class"], "path_2": r2["path"],
            })
    print(f"  Found {len(within_train_flagged)} potential cross-class duplicate(s)")

    # ── Save results ───────────────────────────────────────────────────────────
    with pd.ExcelWriter(OUTPUT_REPORT) as writer:
        if all_flagged:
            pd.DataFrame(all_flagged).sort_values("distance").to_excel(
                writer, sheet_name="Split_Leakage", index=False
            )
        else:
            pd.DataFrame([{"result": "No cross-split near-duplicates found"}]).to_excel(
                writer, sheet_name="Split_Leakage", index=False
            )

        if within_train_flagged:
            pd.DataFrame(within_train_flagged).sort_values("distance").to_excel(
                writer, sheet_name="CrossClass_Duplicates", index=False
            )
        else:
            pd.DataFrame([{"result": "No cross-class duplicates found within train"}]).to_excel(
                writer, sheet_name="CrossClass_Duplicates", index=False
            )

        summary = pd.DataFrame([{
            "dataset": DATASET_NAME, "mode": MODE,
            "train_images": len(train_records),
            "test_images": len(test_records),
            "val_images": len(val_records),
            "hamming_threshold": HAMMING_THRESHOLD,
            "split_leakage_flagged": len(all_flagged),
            "cross_class_duplicates_flagged": len(within_train_flagged),
        }])
        summary.to_excel(writer, sheet_name="Summary", index=False)

    print(f"\n{'='*60}")
    print(f"SUMMARY — {DATASET_NAME}")
    print(f"{'='*60}")
    print(f"  Train images: {len(train_records)}")
    print(f"  Test images:  {len(test_records)}")
    print(f"  Val images:   {len(val_records)}")
    print(f"  Split leakage flagged:          {len(all_flagged)}")
    print(f"  Cross-class duplicates flagged: {len(within_train_flagged)}")
    print(f"\nFull report saved to: {OUTPUT_REPORT}")

#### DATASET & EXPERIMENT INTEGRITY AUDIT  
*(original cell 44)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET & EXPERIMENT INTEGRITY AUDIT
# Verifies, per dataset: (i) class labels, (ii) class balance, (iii) train/val/test
# separation, (iv) random seeds, (v) preprocessing settings, (vi) evaluation metrics.
#
# (i)-(iii) are checked directly against the image folders.
# (iv)-(vi) are checked against the config.txt / results files your training
# pipeline already saves per experiment (see get_model_save_path/save_config
# in your training notebook) — these settings live in your CODE's logs, not
# in the images themselves.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import re
import json
import pandas as pd
from collections import defaultdict

# ── Configure here — run once per dataset ──────────────────────────────────────
DATASET_NAME = "PlantDoc"

# --- (i)-(iii): image folder checks ---
MODE = "split"            # "split" (fixed train/test folders) or "single_pool" (k-fold, e.g. Cassava)
FLAT_STRUCTURE = False    # True if images sit directly in the folder with no class subfolders

TRAIN_DIR = "./plantdoc_local/working/plantdoc_clahe/train"
TEST_DIR  = "./plantdoc_local/working/plantdoc_clahe/test"
VAL_DIR   = None
SINGLE_POOL_DIR = None

EXPECTED_CLASSES = None   # e.g. ["Apple Scab Leaf", "Apple leaf", "Apple rust leaf"] — or None to skip the diff check
IMBALANCE_RATIO_THRESHOLD = 3.0   # flag a split if max-class-count / min-class-count exceeds this

# --- (iv)-(vi): experiment config/results checks ---
EXPERIMENTS_ROOT = "./plantdoc_local/working/plantdoc_experiments"   # scanned for config.txt / results files
EXPERIMENT_NAME_FILTER = None   # e.g. "corn" to only audit experiments with this substring in the folder name; None = all

EXPECTED_SEED = None            # e.g. 42 — set to flag any experiment using a DIFFERENT seed; None = just report what's found
EXPECTED_IMG_SIZE = None        # e.g. "384" — set to flag mismatches; None = just report
EXPECTED_USE_CLAHE = None       # e.g. True — set to flag mismatches; None = just report

REQUIRED_METRICS = ["accuracy", "precision", "recall", "f1"]  # keywords to look for in results files

OUTPUT_REPORT = f"./plantdoc_local/working/integrity_audit_{DATASET_NAME}.xlsx"


# ═══════════════════════════════════════════════════════════════════════════════
# (i) CLASS LABELS + (ii) CLASS BALANCE
# ═══════════════════════════════════════════════════════════════════════════════

def get_class_counts(root_dir, flat=False):
    """Returns {class_name: image_count} for one split folder."""
    counts = {}
    if root_dir is None or not os.path.exists(root_dir):
        return counts
    if flat:
        n = sum(1 for f in os.listdir(root_dir)
                if os.path.isfile(os.path.join(root_dir, f))
                and f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
        counts["N/A (flat structure)"] = n
        return counts
    for cls in sorted(os.listdir(root_dir)):
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        counts[cls] = len([f for f in os.listdir(cls_dir)
                            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
    return counts


def audit_labels_and_balance():
    splits = {}
    if MODE == "single_pool":
        splits["pool"] = get_class_counts(SINGLE_POOL_DIR, flat=FLAT_STRUCTURE)
    else:
        splits["train"] = get_class_counts(TRAIN_DIR, flat=FLAT_STRUCTURE)
        splits["test"] = get_class_counts(TEST_DIR, flat=FLAT_STRUCTURE)
        if VAL_DIR:
            splits["val"] = get_class_counts(VAL_DIR, flat=FLAT_STRUCTURE)

    # --- (i) Class labels ---
    label_rows = []
    all_found_classes = set()
    for split_name, counts in splits.items():
        all_found_classes.update(counts.keys())

    for cls in sorted(all_found_classes):
        row = {"class": cls}
        for split_name in splits:
            row[f"{split_name}_present"] = cls in splits[split_name]
        if EXPECTED_CLASSES is not None:
            row["in_expected_list"] = cls in EXPECTED_CLASSES
        label_rows.append(row)

    missing_from_data = []
    if EXPECTED_CLASSES is not None:
        missing_from_data = [c for c in EXPECTED_CLASSES if c not in all_found_classes]

    # --- (ii) Class balance ---
    balance_rows = []
    for split_name, counts in splits.items():
        if not counts:
            continue
        max_c = max(counts.values())
        min_c = min(counts.values()) if min(counts.values()) > 0 else 1
        ratio = max_c / min_c
        for cls, c in counts.items():
            balance_rows.append({
                "split": split_name, "class": cls, "count": c,
                "split_max_class_count": max_c, "split_min_class_count": min_c,
                "imbalance_ratio": round(ratio, 2),
                "flagged_imbalanced": ratio > IMBALANCE_RATIO_THRESHOLD,
            })

    return label_rows, missing_from_data, balance_rows, splits


# ═══════════════════════════════════════════════════════════════════════════════
# (iii) TRAIN / VAL / TEST SEPARATION
# ═══════════════════════════════════════════════════════════════════════════════

def audit_split_separation(splits_dict_of_dirs):
    """
    Checks for EXACT filename collisions across splits (fast, complements the
    separate perceptual-hash near-duplicate check for near-identical-but-not-
    exact-filename leaks). Also reports split size ratios.
    """
    filename_sets = {}
    for split_name, root_dir in splits_dict_of_dirs.items():
        if root_dir is None or not os.path.exists(root_dir):
            continue
        fnames = set()
        for cls in os.listdir(root_dir):
            cls_dir = os.path.join(root_dir, cls)
            if os.path.isdir(cls_dir):
                fnames.update(os.listdir(cls_dir))
            elif os.path.isfile(cls_dir):
                fnames.add(cls)  # flat structure case
        filename_sets[split_name] = fnames

    collision_rows = []
    names = list(filename_sets.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            s1, s2 = names[i], names[j]
            overlap = filename_sets[s1] & filename_sets[s2]
            collision_rows.append({
                "split_1": s1, "split_2": s2,
                "exact_filename_collisions": len(overlap),
                "example_collisions": ", ".join(list(overlap)[:5]),
            })

    total = sum(len(v) for v in filename_sets.values())
    ratio_rows = [{
        "split": s, "count": len(v),
        "pct_of_total": round(100 * len(v) / total, 1) if total else 0,
    } for s, v in filename_sets.items()]

    return collision_rows, ratio_rows


# ═══════════════════════════════════════════════════════════════════════════════
# (iv) RANDOM SEEDS + (v) PREPROCESSING SETTINGS
# (both read from each experiment's saved config.txt)
# ═══════════════════════════════════════════════════════════════════════════════

def parse_config_txt(filepath):
    """Parses a config.txt (format: 'key: value' per line) into a dict."""
    values = {}
    with open(filepath, "r") as f:
        content = f.read()
    for line in content.splitlines():
        m = re.match(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*:\s*(.+)$", line)
        if m:
            values[m.group(1)] = m.group(2).strip()
    return values


def find_experiment_configs(root, name_filter=None):
    configs = []
    if not os.path.exists(root):
        return configs
    for folder in sorted(os.listdir(root)):
        if name_filter and name_filter.lower() not in folder.lower():
            continue
        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            continue
        cfg_path = os.path.join(folder_path, "config.txt")
        if os.path.exists(cfg_path):
            configs.append((folder, cfg_path))
    return configs


def audit_seeds_and_preprocessing():
    configs = find_experiment_configs(EXPERIMENTS_ROOT, EXPERIMENT_NAME_FILTER)

    seed_rows = []
    preprocess_rows = []

    for exp_name, cfg_path in configs:
        cfg = parse_config_txt(cfg_path)

        seed_val = cfg.get("random_seed", "MISSING")
        seed_rows.append({
            "experiment": exp_name,
            "random_seed": seed_val,
            "matches_expected": (str(seed_val) == str(EXPECTED_SEED)) if EXPECTED_SEED is not None else "N/A",
            "flagged_missing": seed_val == "MISSING",
        })

        img_size = cfg.get("img_size", "MISSING")
        use_clahe = cfg.get("use_clahe", "MISSING")
        preprocess_rows.append({
            "experiment": exp_name,
            "img_size": img_size,
            "use_clahe": use_clahe,
            "clahe_clip_limit": cfg.get("clahe_clip_limit", "MISSING"),
            "clahe_tile_grid": cfg.get("clahe_tile_grid", "MISSING"),
            "use_augmentation": cfg.get("use_augmentation", "MISSING"),
            "use_cutmix": cfg.get("use_cutmix", "MISSING"),
            "use_mixup": cfg.get("use_mixup", "MISSING"),
            "img_size_matches_expected": (str(img_size) == str(EXPECTED_IMG_SIZE)) if EXPECTED_IMG_SIZE is not None else "N/A",
            "clahe_matches_expected": (str(use_clahe) == str(EXPECTED_USE_CLAHE)) if EXPECTED_USE_CLAHE is not None else "N/A",
        })

    return seed_rows, preprocess_rows, len(configs)


# ═══════════════════════════════════════════════════════════════════════════════
# (vi) EVALUATION METRICS
# Scans every .txt / .json in each experiment folder for the required metric
# keywords, reporting which are present / missing per experiment.
# ═══════════════════════════════════════════════════════════════════════════════

def audit_evaluation_metrics():
    metric_rows = []
    if not os.path.exists(EXPERIMENTS_ROOT):
        return metric_rows

    for folder in sorted(os.listdir(EXPERIMENTS_ROOT)):
        if EXPERIMENT_NAME_FILTER and EXPERIMENT_NAME_FILTER.lower() not in folder.lower():
            continue
        folder_path = os.path.join(EXPERIMENTS_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue

        combined_text = ""
        for fname in os.listdir(folder_path):
            if fname.endswith((".txt", ".json")):
                try:
                    with open(os.path.join(folder_path, fname), "r") as f:
                        combined_text += f.read().lower()
                except Exception:
                    continue

        row = {"experiment": folder}
        for metric in REQUIRED_METRICS:
            row[f"has_{metric}"] = metric in combined_text
        row["all_required_present"] = all(row[f"has_{m}"] for m in REQUIRED_METRICS)
        metric_rows.append(row)

    return metric_rows


# ═══════════════════════════════════════════════════════════════════════════════
# RUN EVERYTHING
# ═══════════════════════════════════════════════════════════════════════════════

print(f"Running integrity audit for: {DATASET_NAME}\n")

print("(i)+(ii) Auditing class labels and balance...")
label_rows, missing_classes, balance_rows, splits = audit_labels_and_balance()
n_imbalanced = sum(1 for r in balance_rows if r["flagged_imbalanced"])
print(f"  Classes found: {len(label_rows)} | Missing from expected list: {len(missing_classes)} | Imbalanced (class,split) pairs flagged: {n_imbalanced}")

print("\n(iii) Auditing train/val/test separation...")
split_dirs = {"pool": SINGLE_POOL_DIR} if MODE == "single_pool" else {
    "train": TRAIN_DIR, "test": TEST_DIR, **({"val": VAL_DIR} if VAL_DIR else {})
}
collision_rows, ratio_rows = audit_split_separation(split_dirs)
n_collisions = sum(r["exact_filename_collisions"] for r in collision_rows)
print(f"  Exact filename collisions across splits: {n_collisions}")

print("\n(iv)+(v) Auditing random seeds and preprocessing settings...")
seed_rows, preprocess_rows, n_experiments = audit_seeds_and_preprocessing()
n_missing_seed = sum(1 for r in seed_rows if r["flagged_missing"])
print(f"  Experiments scanned: {n_experiments} | Missing random_seed: {n_missing_seed}")

print("\n(vi) Auditing evaluation metrics...")
metric_rows = audit_evaluation_metrics()
n_incomplete = sum(1 for r in metric_rows if not r["all_required_present"])
print(f"  Experiments scanned: {len(metric_rows)} | Missing required metrics: {n_incomplete}")

# ── Save full report ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_REPORT) as writer:
    pd.DataFrame(label_rows).to_excel(writer, sheet_name="i_Class_Labels", index=False)
    if missing_classes:
        pd.DataFrame({"missing_expected_class": missing_classes}).to_excel(
            writer, sheet_name="i_Missing_Classes", index=False)
    pd.DataFrame(balance_rows).to_excel(writer, sheet_name="ii_Class_Balance", index=False)
    pd.DataFrame(collision_rows).to_excel(writer, sheet_name="iii_Split_Collisions", index=False)
    pd.DataFrame(ratio_rows).to_excel(writer, sheet_name="iii_Split_Ratios", index=False)
    pd.DataFrame(seed_rows).to_excel(writer, sheet_name="iv_Random_Seeds", index=False)
    pd.DataFrame(preprocess_rows).to_excel(writer, sheet_name="v_Preprocessing", index=False)
    pd.DataFrame(metric_rows).to_excel(writer, sheet_name="vi_Evaluation_Metrics", index=False)

    summary = pd.DataFrame([{
        "dataset": DATASET_NAME,
        "classes_found": len(label_rows),
        "classes_missing_from_expected": len(missing_classes),
        "imbalanced_class_split_pairs": n_imbalanced,
        "exact_filename_collisions": n_collisions,
        "experiments_scanned": n_experiments,
        "experiments_missing_seed": n_missing_seed,
        "experiments_missing_required_metrics": n_incomplete,
    }])
    summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"\n{'='*60}")
print(f"Full report saved to: {OUTPUT_REPORT}")
print(f"{'='*60}")

#### SPECIALIST GROUP IMAGE COUNTS — plantdoc_clahe  
*(original cell 45)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPECIALIST GROUP IMAGE COUNTS — plantdoc_clahe
# Scans the current (post-cleaning, post-remediation) plantdoc_clahe dataset and
# counts valid images per specialist group: Apple, Bell Pepper, Corn, Potato,
# Tomato, Grape, Rest. Use these counts as the weights in the theoretical
# pipeline accuracy formula, so weighting reflects the ACTUAL current dataset
# size (after artifact removal + duplicate-contamination remediation), not the
# original pre-cleaning counts.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

# ── Configure here ────────────────────────────────────────────────────────────
TRAIN_DIR = "./plantdoc_local/working/plantdoc_clahe/train"
TEST_DIR  = "./plantdoc_local/working/plantdoc_clahe/test"
OUTPUT_REPORT = "./plantdoc_local/working/specialist_group_counts.xlsx"

# Keyword-based grouping — matches the same keyword-filtering logic already
# used by build_filtered_dataset() elsewhere in the pipeline. Matching is
# case-insensitive substring matching against each class folder name.
GROUP_KEYWORDS = {
    "Apple":       ["apple"],
    "Bell Pepper": ["bell_pepper", "bell pepper"],
    "Corn":        ["corn"],
    "Potato":      ["potato"],
    "Tomato":      ["tomato"],
    "Grape":       ["grape"],
    # Rest = everything NOT matched by the above keywords (Blueberry, Cherry,
    # Peach, Raspberry, Soybean, Squash, Strawberry)
}


def classify_folder(folder_name):
    """Returns the specialist group name for a given class folder name."""
    name_lower = folder_name.lower()
    for group, keywords in GROUP_KEYWORDS.items():
        if any(kw in name_lower for kw in keywords):
            return group
    return "Rest"


def count_images_in_dir(root_dir):
    """Returns {group: {class_folder: count}} for one split directory."""
    results = {}
    if not os.path.exists(root_dir):
        print(f"  WARNING: directory not found: {root_dir}")
        return results

    for folder in sorted(os.listdir(root_dir)):
        folder_path = os.path.join(root_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        n_images = len([
            f for f in os.listdir(folder_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
        ])
        group = classify_folder(folder)
        results.setdefault(group, {})[folder] = n_images

    return results


def summarize(results, split_name):
    """Builds a flat list of rows and a group-total dict from count_images_in_dir output."""
    rows = []
    group_totals = {}
    for group, class_counts in results.items():
        group_total = sum(class_counts.values())
        group_totals[group] = group_total
        for cls, count in class_counts.items():
            rows.append({
                "split": split_name, "group": group, "class": cls, "count": count,
            })
    return rows, group_totals


# ── Run for both splits ─────────────────────────────────────────────────────────
print("Scanning train...")
train_results = count_images_in_dir(TRAIN_DIR)
train_rows, train_totals = summarize(train_results, "train")

print("Scanning test...")
test_results = count_images_in_dir(TEST_DIR)
test_rows, test_totals = summarize(test_results, "test")

# ── Print summary ────────────────────────────────────────────────────────────
all_groups = ["Apple", "Bell Pepper", "Corn", "Potato", "Tomato", "Grape", "Rest"]

print(f"\n{'='*60}")
print(f"SPECIALIST GROUP COUNTS (post-cleaning, post-remediation)")
print(f"{'='*60}")
print(f"{'Group':<14}{'Train':>10}{'Test':>10}{'Total':>10}")
for group in all_groups:
    tr = train_totals.get(group, 0)
    te = test_totals.get(group, 0)
    print(f"{group:<14}{tr:>10}{te:>10}{tr+te:>10}")

train_total_all = sum(train_totals.values())
test_total_all = sum(test_totals.values())
print(f"{'TOTAL':<14}{train_total_all:>10}{test_total_all:>10}{train_total_all+test_total_all:>10}")

# ── Save full report ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_REPORT) as writer:
    pd.DataFrame(train_rows).to_excel(writer, sheet_name="Train_PerClass", index=False)
    pd.DataFrame(test_rows).to_excel(writer, sheet_name="Test_PerClass", index=False)

    summary_rows = []
    for group in all_groups:
        summary_rows.append({
            "group": group,
            "train_count": train_totals.get(group, 0),
            "test_count": test_totals.get(group, 0),
            "total_count": train_totals.get(group, 0) + test_totals.get(group, 0),
        })
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Group_Summary", index=False)

print(f"\nFull report saved to: {OUTPUT_REPORT}")
print(f"\nUse the 'test_count' column as your pipeline-accuracy weighting")
print(f"(matches the original formula's use of validation/test set sizes as N_i).")

#### COUNT CLASSIFIER TEST IMAGES  
*(original cell 49)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COUNT CLASSIFIER TEST IMAGES
# Checks whether the identical val_accuracy values across seeds (0.97484277)
# are explainable by test-set quantization — i.e. whether some small integer
# N gives exactly correct/N = 0.97484277.
# ═══════════════════════════════════════════════════════════════════════════════

import os

CLASSIFIER_TEST_DIR = "/home/auc/PlantDoc/Datasets/crop_classifier_v5/test"

total = 0
per_species = {}
for species_folder in sorted(os.listdir(CLASSIFIER_TEST_DIR)):
    species_path = os.path.join(CLASSIFIER_TEST_DIR, species_folder)
    if not os.path.isdir(species_path):
        continue
    n = len([
        f for f in os.listdir(species_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ])
    per_species[species_folder] = n
    total += n

print("Per-species test counts:")
for species, n in per_species.items():
    print(f"  {species}: {n}")
print(f"\nTotal classifier test images: {total}")

# ── Check whether 0.97484277 matches an exact fraction of this total ──────────
target_accuracy = 0.97484277
print(f"\nChecking which integer correct-count / {total} is closest to {target_accuracy}...")
best_diff = float("inf")
best_correct = None
for correct in range(total + 1):
    frac = correct / total
    diff = abs(frac - target_accuracy)
    if diff < best_diff:
        best_diff = diff
        best_correct = correct

print(f"  Closest match: {best_correct}/{total} = {best_correct/total:.8f}")
print(f"  Target was:              {target_accuracy:.8f}")
print(f"  Difference: {best_diff:.8f}")

if best_diff < 1e-6:
    print("\n  EXACT MATCH — confirms this is test-set quantization, not a bug.")
    print(f"  4 of 5 seeds all correctly classified exactly {best_correct} of {total} images.")
else:
    print("\n  NOT an exact match — worth investigating further, since the identical")
    print("  values across seeds aren't explained by simple quantization at this N.")

## B. Copy-Paste V2 (Phase 3b) — UPDATE BEFORE RE-RUNNING

These cells use the v1 protocol. Before re-running: generate composites from the new `train/` only, use the protocol-v2 runner, and read `test_accuracy` instead of `val_accuracy`. Cell *reuse_none_baseline* is obsolete under v2: the `none` condition is the main run itself.

#### LESION BOX DETECTION + SIZE MEASUREMENT  
*(original cell 50)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LESION BOX DETECTION + SIZE MEASUREMENT
# Detects your colored marker boxes (drawn manually around lesions), converts
# each to pixel coordinates, and computes the REAL lesion size (both in pixels
# and as a percentage of the image) — replacing the old assumed 10-20% patch
# size with a measured value, per specialist.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
from PIL import Image
import pandas as pd
import cv2

# ── Configure here ────────────────────────────────────────────────────────────
# One entry per specialist you've annotated. ANNOTATED_DIR = images with your
# drawn boxes. ORIGINAL_DIR = the matching clean (unannotated) source images.
SPECIALIST_DIRS = {
    "Apple":       {"original": "/home/auc/PlantDoc/Experiments/filtered_Apple/train",
                     "annotated":  "/home/auc/PlantDoc/Datasets/Annotated Images/Apple_to_annotate (1)"},
    "Bell_pepper": {"original": "/home/auc/PlantDoc/Experiments/filtered_Bell_pepper/train",
                     "annotated":  "/home/auc/PlantDoc/Datasets/Annotated Images/Bell_pepper_to_annotate (1)"},
    "Tomato":      {"original": "/home/auc/PlantDoc/Experiments/filtered_Tomato/train",
                     "annotated":  "/home/auc/PlantDoc/Datasets/Annotated Images/Tomato_to_annotate (1)"},
    "Grape":       {"original": "/home/auc/PlantDoc/Experiments/filtered_Grape/train",
                     "annotated":  "/home/auc/PlantDoc/Datasets/Annotated Images/Grape_to_annotate (1)"},
}

MARKER_COLOR = (220, 40, 35)   # corrected from pure (255,0,0) — actual drawn/compressed
                                  # color measured directly from a real annotated sample
COLOR_TOLERANCE = 60             # widened from 30 — JPEG compression spreads the box's
                                  # color significantly (observed range: R 151-237,
                                  # G 29-98, B 24-96 in the sample checked)

OUTPUT_CSV = "/home/auc/PlantDoc/lesion_coordinates.csv"


def find_marker_box(annotated_path, color, tolerance,
                     min_area_pct=0.3, min_rectangularity=0.7,
                     min_aspect=0.5, max_aspect=2.0):
    """
    Detects your drawn SQUARE/RECTANGLE marker by shape, not just color —
    critical when the marker color can also appear naturally elsewhere in
    the image (e.g. a red apple, reddish-brown lesions). Finds all separate
    color-matching regions, then keeps only the one that actually looks like
    a drawn rectangle: high "rectangularity" (contour area / bounding box
    area — a drawn outline traces a near-complete rectangle, while organic
    shapes like fruit skin or scattered noise pixels do not) and a roughly
    square-ish aspect ratio.
    """
    img = np.array(Image.open(annotated_path).convert("RGB"))
    diff = np.abs(img.astype(int) - np.array(color).astype(int))
    mask = np.all(diff <= tolerance, axis=-1).astype(np.uint8) * 255

    if not mask.any():
        return None, img.shape

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, img.shape

    img_area = img.shape[0] * img.shape[1]
    candidates = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        bbox_area = w * h
        if bbox_area == 0:
            continue
        area_pct = 100 * bbox_area / img_area
        aspect = w / h
        rectangularity = cv2.contourArea(cnt) / bbox_area

        if (area_pct >= min_area_pct and
                rectangularity >= min_rectangularity and
                min_aspect <= aspect <= max_aspect):
            candidates.append((x, y, x + w, y + h, rectangularity, area_pct))

    if not candidates:
        return None, img.shape

    # If multiple candidates pass the filter, take the one with the highest
    # rectangularity (most likely to be your actual drawn box, not a
    # coincidental near-square false match)
    candidates.sort(key=lambda c: c[4], reverse=True)
    x1, y1, x2, y2, _, _ = candidates[0]
    return (x1, y1, x2, y2), img.shape


def find_original_image(fname, original_dir):
    """Locates the matching unannotated original, searching subfolders."""
    for root, _, files in os.walk(original_dir):
        if fname in files:
            return os.path.join(root, fname)
    return None


rows = []
for specialist, dirs in SPECIALIST_DIRS.items():
    annotated_dir = dirs["annotated"]
    original_dir = dirs["original"]

    if not os.path.exists(annotated_dir):
        print(f"  SKIP {specialist}: annotated dir not found ({annotated_dir})")
        continue

    # Walk RECURSIVELY — your annotated folder likely still has the same
    # per-disease-class subfolder structure as the original download
    # (e.g. Apple_to_annotate/Apple Scab Leaf/img1.jpg), not a flat list
    # of images directly inside annotated_dir.
    annotated_files = []
    for root, _, files in os.walk(annotated_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                annotated_files.append(os.path.join(root, f))

    if not annotated_files:
        print(f"  WARNING: {specialist} — annotated_dir exists but contains "
              f"NO image files anywhere inside it (searched recursively): {annotated_dir}")

    for annotated_path in sorted(annotated_files):
        fname = os.path.basename(annotated_path)
        box, img_shape = find_marker_box(
                annotated_path, MARKER_COLOR, COLOR_TOLERANCE,
                min_aspect=0.2, max_aspect=5.0   # allows rectangles up to 5x longer than tall (or vice versa)
            )

        if box is None:
            print(f"  WARNING: no marker found in {specialist}/{fname}")
            continue

        x1, y1, x2, y2 = box
        img_h, img_w = img_shape[0], img_shape[1]

        lesion_w = x2 - x1
        lesion_h = y2 - y1
        lesion_area = lesion_w * lesion_h
        image_area = img_w * img_h
        area_pct = 100 * lesion_area / image_area

        original_path = find_original_image(fname, original_dir)
        if original_path is None:
            print(f"  WARNING: original not found for {specialist}/{fname}")

        rows.append({
            "specialist": specialist,
            "filename": fname,
            "original_path": original_path,
            "annotated_path": annotated_path,
            "image_width": img_w, "image_height": img_h,
            "x_min": x1, "y_min": y1, "x_max": x2, "y_max": y2,
            "lesion_width_px": lesion_w, "lesion_height_px": lesion_h,
            "lesion_area_px": lesion_area,
            "lesion_area_pct_of_image": round(area_pct, 3),
            "lesion_width_pct_of_image": round(100 * lesion_w / img_w, 3),
            "lesion_height_pct_of_image": round(100 * lesion_h / img_h, 3),
        })

if not rows:
    raise RuntimeError(
        "No lesion boxes were detected across ANY specialist. Check the "
        "warnings printed above this line:\n"
        "  - 'annotated dir not found' means SPECIALIST_DIRS paths are wrong "
        "(still a placeholder, or a typo)\n"
        "  - 'no marker found' means the path is right but MARKER_COLOR/"
        "COLOR_TOLERANCE isn't matching your drawn boxes — double check the "
        "exact RGB color you drew in, and that the image wasn't re-compressed "
        "in a way that shifted the color."
    )

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'='*60}")
print(f"LESION SIZE SUMMARY (measured from real annotations)")
print(f"{'='*60}")
for specialist in df["specialist"].unique():
    sub = df[df["specialist"] == specialist]
    print(f"\n{specialist} (n={len(sub)}):")
    print(f"  Lesion area  % of image: mean={sub['lesion_area_pct_of_image'].mean():.2f}%  "
          f"median={sub['lesion_area_pct_of_image'].median():.2f}%  "
          f"range=[{sub['lesion_area_pct_of_image'].min():.2f}%, {sub['lesion_area_pct_of_image'].max():.2f}%]")
    print(f"  Lesion width % of image: mean={sub['lesion_width_pct_of_image'].mean():.2f}%")
    print(f"  Lesion height % of image: mean={sub['lesion_height_pct_of_image'].mean():.2f}%")

# ── Flag suspicious outliers for manual review ─────────────────────────────────
# Very small areas may be spurious tiny color matches elsewhere in the image;
# very large areas are almost certainly false-positive detections (the widened
# tolerance catching a large naturally reddish/brown leaf region instead of
# your drawn box) rather than a genuinely huge annotated lesion.
LOW_THRESHOLD = 1
HIGH_THRESHOLD = 60

flagged = df[(df["lesion_area_pct_of_image"] < LOW_THRESHOLD) |
             (df["lesion_area_pct_of_image"] > HIGH_THRESHOLD)]

print(f"\n{'='*60}")
print(f"FLAGGED FOR MANUAL REVIEW (area < {LOW_THRESHOLD}% or > {HIGH_THRESHOLD}%)")
print(f"{'='*60}")
print(f"{len(flagged)} of {len(df)} images flagged ({100*len(flagged)/len(df):.1f}%)\n")

for specialist in flagged["specialist"].unique():
    sub = flagged[flagged["specialist"] == specialist]
    print(f"{specialist}: {len(sub)} flagged")
    for _, row in sub.iterrows():
        direction = "TOO SMALL" if row["lesion_area_pct_of_image"] < LOW_THRESHOLD else "TOO LARGE"
        print(f"  [{direction}] {row['filename']}: {row['lesion_area_pct_of_image']:.2f}%  "
              f"(box: {row['x_min']},{row['y_min']} to {row['x_max']},{row['y_max']})")
        print(f"      annotated: {row['annotated_path']}")

flagged_csv = OUTPUT_CSV.replace(".csv", "_flagged.csv")
flagged.to_csv(flagged_csv, index=False)
print(f"\nFlagged rows also saved separately to: {flagged_csv}")
print("Open the 'annotated_path' for each flagged image and visually confirm")
print("whether the detected box is correct — especially the TOO LARGE ones,")
print("which likely indicate a false-positive match, not a real giant lesion.")

print(f"\nFull coordinates + measurements saved to: {OUTPUT_CSV}")
print("\nThis replaces the old assumed 10-20% patch_ratio with real, measured")
print("values — use the per-specialist mean/median area % below as your new")
print("patch_ratio basis in the redesigned copy_paste function.")

#### COPY-PASTE V2 — OFFLINE GENERATOR  
*(original cell 51)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COPY-PASTE V2 — OFFLINE GENERATOR
# Pre-generates composite images for each of the 5 ablation conditions, as
# real saved files — not a live training-time augmentation. This is the right
# design for this technique: real per-image lesion lookup by filename is
# fundamentally incompatible with a live tf.data batch pipeline, which only
# sees raw tensors, not filenames, once running.
#
# Workflow: run this ONCE per (specialist, condition) pair to generate a
# folder of composite images, then point CONFIG["dataset_dir_train"] at a
# combined folder (originals + that condition's composites) for that
# training run. Repeat per condition for the ablation ladder.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import cv2
import pandas as pd
from PIL import Image

DEBUG_PLACEMENT = True  # prints diagnostic info when the fallback placement also fails

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LEAF SEGMENTATION (background-aware placement)
# ═══════════════════════════════════════════════════════════════════════════════

def get_leaf_mask(image_rgb, margin_px=15):
    """Binary mask: 255 = valid leaf placement area, 0 = background/near-edge."""
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    lower = np.array([5, 30, 30])
    upper = np.array([100, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    h, w = mask.shape
    mask[:margin_px, :] = 0
    mask[-margin_px:, :] = 0
    mask[:, :margin_px] = 0
    mask[:, -margin_px:] = 0
    return mask


def find_valid_placement_center(leaf_mask, patch_h, patch_w, max_attempts=50, rng=None):
    """
    Finds a placement whose patch footprint lies entirely within valid leaf
    area. Tries random search first (fast, and produces varied placement
    across images); if that fails after max_attempts, falls back to a
    deterministic check against the LARGEST connected leaf region specifically
    — this catches cases where a valid placement genuinely exists (e.g. a
    large, well-formed leaf blob) but random sampling simply didn't stumble
    onto it in max_attempts tries, which becomes more likely as patch size
    grows relative to the image (large patches need to land in a fairly
    specific, big-enough contiguous region).
    """
    rng = rng or np.random
    h, w = leaf_mask.shape

    # ── Stage 1: random search (as before) ─────────────────────────────────
    ys, xs = np.where(leaf_mask > 0)
    if len(ys) > 0:
        for _ in range(max_attempts):
            idx = rng.randint(0, len(ys))
            cy, cx = ys[idx], xs[idx]
            y1, y2 = cy - patch_h // 2, cy - patch_h // 2 + patch_h
            x1, x2 = cx - patch_w // 2, cx - patch_w // 2 + patch_w
            if y1 < 0 or x1 < 0 or y2 >= h or x2 >= w:
                continue
            if leaf_mask[y1:y2, x1:x2].mean() > 200:
                return (cy, cx)

    # ── Stage 2: deterministic fallback — largest connected component ──────
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        (leaf_mask > 0).astype(np.uint8), connectivity=8
    )
    if num_labels <= 1:
        return None  # no leaf regions at all

    # Skip label 0 (background); find the largest actual leaf component
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    comp_mask = (labels == largest_label).astype(np.uint8) * 255
    comp_w, comp_h = stats[largest_label, cv2.CC_STAT_WIDTH], stats[largest_label, cv2.CC_STAT_HEIGHT]
    comp_area = stats[largest_label, cv2.CC_STAT_AREA]
    total_leaf_area = int((leaf_mask > 0).sum())

    # Try placing centered on this component's centroid, and a small local
    # search around it, since the centroid itself might not have full
    # patch-footprint clearance even if the component overall is big enough
    cy0, cx0 = int(centroids[largest_label][1]), int(centroids[largest_label][0])
    for dy in range(-20, 21, 5):
        for dx in range(-20, 21, 5):
            cy, cx = cy0 + dy, cx0 + dx
            y1, y2 = cy - patch_h // 2, cy - patch_h // 2 + patch_h
            x1, x2 = cx - patch_w // 2, cx - patch_w // 2 + patch_w
            if y1 < 0 or x1 < 0 or y2 >= h or x2 >= w:
                continue
            if comp_mask[y1:y2, x1:x2].mean() > 200:
                return (cy, cx)

    if DEBUG_PLACEMENT:
        print(f"    [DEBUG] Fallback failed: patch needs {patch_h}x{patch_w}, "
              f"largest component bbox is {comp_h}x{comp_w} (area={comp_area}px), "
              f"num_components={num_labels-1}, total_leaf_area={total_leaf_area}px "
              f"({'FRAGMENTED — many small islands' if comp_area < total_leaf_area * 0.5 and num_labels > 3 else 'one dominant region, genuinely too small'})")

    return None  # even the largest component can't fit this patch — genuinely skip


# ═══════════════════════════════════════════════════════════════════════════════
# 2. COLOR + ILLUMINATION MATCHING
# ═══════════════════════════════════════════════════════════════════════════════

def match_color_and_illumination(patch_rgb, destination_region_rgb, color_match_strength=0.25):
    """
    Matches ILLUMINATION (the L/lightness channel) fully between patch and
    destination — this corrects for genuinely different lighting/exposure
    conditions between two separate source photos.

    The A/B (color) channels get only a PARTIAL correction (controlled by
    color_match_strength, default 0.25 = 25% shift toward destination stats).
    Full color matching would shift the patch's color to statistically match
    the destination — which, for a diseased patch pasted onto healthy tissue,
    mathematically erases the lesion's actual color identity (a brown lesion
    color-matched to a green background becomes indistinguishable from green).
    Partial matching softens harsh color-temperature mismatches between the
    two source photos while preserving the diagnostic color signal that makes
    this technique meaningful in the first place.
    """
    patch_lab = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    dest_lab = cv2.cvtColor(destination_region_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

    matched = patch_lab.copy()

    # L channel (lightness/illumination) — FULL matching
    p_mean, p_std = patch_lab[..., 0].mean(), patch_lab[..., 0].std() + 1e-6
    d_mean, d_std = dest_lab[..., 0].mean(), dest_lab[..., 0].std() + 1e-6
    matched[..., 0] = (patch_lab[..., 0] - p_mean) * (d_std / p_std) + d_mean

    # A, B channels (color) — PARTIAL matching only, preserving lesion color identity
    for c in [1, 2]:
        p_mean, p_std = patch_lab[..., c].mean(), patch_lab[..., c].std() + 1e-6
        d_mean, d_std = dest_lab[..., c].mean(), dest_lab[..., c].std() + 1e-6
        fully_matched = (patch_lab[..., c] - p_mean) * (d_std / p_std) + d_mean
        # Blend between the ORIGINAL patch color and the fully-matched color,
        # by color_match_strength — preserves most of the original lesion hue
        matched[..., c] = (1 - color_match_strength) * patch_lab[..., c] + color_match_strength * fully_matched

    matched = np.clip(matched, 0, 255).astype(np.uint8)
    return cv2.cvtColor(matched, cv2.COLOR_LAB2RGB)


# ═══════════════════════════════════════════════════════════════════════════════
# 3. EDGE FEATHERING
# ═══════════════════════════════════════════════════════════════════════════════

def make_feathered_alpha_mask(patch_h, patch_w, feather_px=8):
    alpha = np.ones((patch_h, patch_w), dtype=np.float32)
    feather_px = min(feather_px, patch_h // 2, patch_w // 2)
    for i in range(feather_px):
        v = (i + 1) / feather_px
        alpha[i, :] = np.minimum(alpha[i, :], v)
        alpha[-(i + 1), :] = np.minimum(alpha[-(i + 1), :], v)
        alpha[:, i] = np.minimum(alpha[:, i], v)
        alpha[:, -(i + 1)] = np.minimum(alpha[:, -(i + 1)], v)
    return alpha


def composite_patch(base_image_rgb, patch_rgb, center_y, center_x,
                     blended=True, match_appearance=True, feather_px=8):
    result = base_image_rgb.copy()
    ph, pw = patch_rgb.shape[:2]
    y1, y2 = center_y - ph // 2, center_y - ph // 2 + ph
    x1, x2 = center_x - pw // 2, center_x - pw // 2 + pw
    destination_region = base_image_rgb[y1:y2, x1:x2]

    working_patch = patch_rgb.copy()
    if match_appearance:
        working_patch = match_color_and_illumination(working_patch, destination_region)

    if blended:
        alpha = make_feathered_alpha_mask(ph, pw, feather_px=feather_px)[..., np.newaxis]
        composited = (working_patch.astype(np.float32) * alpha +
                      destination_region.astype(np.float32) * (1 - alpha))
        result[y1:y2, x1:x2] = np.clip(composited, 0, 255).astype(np.uint8)
    else:
        result[y1:y2, x1:x2] = working_patch

    return result


# ═══════════════════════════════════════════════════════════════════════════════
# 4. MAIN GENERATOR
# ═══════════════════════════════════════════════════════════════════════════════

def generate_condition(specialist, condition, lesion_csv_path, healthy_dir,
                        diseased_dir, output_dir, seed, n_per_healthy_image=1,
                        default_patch_ratio=0.15):
    """
    Generates composite images for ONE (specialist, condition) pair.

    condition: "lesion_unblended" | "lesion_blended" | "healthy_patch" |
               "random_nonlesion"   ("none" = skip, no generation needed)
    """
    rng = np.random.RandomState(seed)
    os.makedirs(output_dir, exist_ok=True)

    lesion_df = pd.read_csv(lesion_csv_path)
    lesion_df = lesion_df[lesion_df["specialist"] == specialist]
    lesion_lookup = {
        row["filename"]: (row["x_min"], row["y_min"], row["x_max"], row["y_max"])
        for _, row in lesion_df.iterrows()
    }
    measured_ratio = (lesion_df["lesion_area_pct_of_image"].mean() / 100
                       if len(lesion_df) else default_patch_ratio)

    healthy_files = [f for f in os.listdir(healthy_dir)
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    diseased_files = [f for f in os.listdir(diseased_dir)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if condition in ("lesion_unblended", "lesion_blended"):
        # Only use diseased images that were actually annotated with a real lesion box
        diseased_files = [f for f in diseased_files if f in lesion_lookup]
        if not diseased_files:
            print(f"  WARNING: no annotated lesions found for {specialist} — skipping {condition}")
            return 0

    generated = 0
    skip_reasons = {"degenerate_patch": 0, "no_valid_placement": 0, "empty_leaf_mask": 0}
    failed_filenames = []   # actual filenames that failed, for manual inspection
    for healthy_fname in healthy_files:
        healthy_path = os.path.join(healthy_dir, healthy_fname)
        base_img = np.array(Image.open(healthy_path).convert("RGB"))
        leaf_mask = get_leaf_mask(base_img)

        if leaf_mask.sum() == 0:
            skip_reasons["empty_leaf_mask"] += n_per_healthy_image
            failed_filenames.append((healthy_fname, "empty_leaf_mask"))
            continue  # no leaf area detected at all in this image — skip entirely

        for rep in range(n_per_healthy_image):
            src_fname = diseased_files[rng.randint(0, len(diseased_files))]
            src_path = os.path.join(diseased_dir, src_fname)
            src_img = np.array(Image.open(src_path).convert("RGB"))

            if condition in ("lesion_unblended", "lesion_blended"):
                x1, y1, x2, y2 = lesion_lookup[src_fname]
                patch = src_img[y1:y2, x1:x2]
            elif condition == "random_nonlesion":
                ph = max(8, int(src_img.shape[0] * np.sqrt(measured_ratio)))
                pw = max(8, int(src_img.shape[1] * np.sqrt(measured_ratio)))
                sy = rng.randint(0, max(1, src_img.shape[0] - ph))
                sx = rng.randint(0, max(1, src_img.shape[1] - pw))
                patch = src_img[sy:sy+ph, sx:sx+pw]
            elif condition == "healthy_patch":
                other_healthy_fname = healthy_files[rng.randint(0, len(healthy_files))]
                other_img = np.array(Image.open(os.path.join(healthy_dir, other_healthy_fname)).convert("RGB"))
                ph = max(8, int(other_img.shape[0] * np.sqrt(measured_ratio)))
                pw = max(8, int(other_img.shape[1] * np.sqrt(measured_ratio)))
                sy = rng.randint(0, max(1, other_img.shape[0] - ph))
                sx = rng.randint(0, max(1, other_img.shape[1] - pw))
                patch = other_img[sy:sy+ph, sx:sx+pw]
            else:
                continue

            if patch.shape[0] < 4 or patch.shape[1] < 4:
                skip_reasons["degenerate_patch"] += 1
                continue  # degenerate patch, skip

            placement = find_valid_placement_center(leaf_mask, patch.shape[0], patch.shape[1], rng=rng)
            if placement is None:
                skip_reasons["no_valid_placement"] += 1
                failed_filenames.append((healthy_fname, "no_valid_placement"))
                continue

            cy, cx = placement
            blended = condition == "lesion_blended"
            composite = composite_patch(base_img, patch, cy, cx,
                                          blended=blended, match_appearance=blended)

            out_fname = f"{condition}__{healthy_fname.rsplit('.',1)[0]}__rep{rep}.jpg"
            Image.fromarray(composite).save(os.path.join(output_dir, out_fname), quality=95)
            generated += 1

    total_attempted = len(healthy_files) * n_per_healthy_image
    print(f"  {specialist} / {condition}: generated {generated} of {total_attempted} attempted -> {output_dir}")
    print(f"    Skipped: {skip_reasons['empty_leaf_mask']} (no leaf detected at all), "
          f"{skip_reasons['no_valid_placement']} (leaf detected but no valid placement found), "
          f"{skip_reasons['degenerate_patch']} (patch too small)")
    if failed_filenames:
        print(f"    Failed filenames (for manual inspection — check if these are "
              f"genuinely too-small/distant/close-cropped leaf photos):")
        for fname, reason in failed_filenames:
            print(f"      [{reason}] {fname}")
    return generated


# ── Example usage — run once per (specialist, condition) pair ─────────────────
# if __name__ == "__main__":
#     SPECIALISTS = ["Grape"]
#     CONDITIONS = ["lesion_unblended", "lesion_blended", "healthy_patch", "random_nonlesion"]
#     SEED = 42
#     LESION_CSV = "/home/auc/PlantDoc/lesion_coordinates.csv"

#     for specialist in SPECIALISTS:
#         base = f"/home/auc/PlantDoc/Experiments/filtered_{specialist}/train"
#         # NOTE: adjust these healthy/diseased class folder names per specialist
#         healthy_dir = os.path.join(base, "grape leaf")
#         diseased_dir = os.path.join(base, "grape leaf black rot")

#         for condition in CONDITIONS:
#             output_dir = f"./plantdoc_local/working/copy_paste_v2/{specialist}/{condition}"
#             generate_condition(
#                 specialist=specialist, condition=condition,
#                 lesion_csv_path=LESION_CSV,
#                 healthy_dir=healthy_dir, diseased_dir=diseased_dir,
#                 output_dir=output_dir, seed=SEED,
#             )

#### COPY-PASTE V2 — FULL ABLATION LADDER DRIVER  
*(original cell 55)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COPY-PASTE V2 — FULL ABLATION LADDER DRIVER
# Runs all 5 conditions (none, lesion_unblended, lesion_blended, healthy_patch,
# random_nonlesion) through your existing multi-seed infrastructure, for a
# given specialist. Reuses run_multi_seed_experiment() and the resume_path
# mechanism unchanged — no changes to your existing multi-seed code needed.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import shutil
import json

# ── Configure here — one specialist per run ────────────────────────────────────
# Reference — all 4 Copy-Paste-eligible specialists' class folder names:
#   Apple:       healthy="Apple leaf"        diseased=["Apple Scab Leaf", "Apple rust leaf"]
#   Bell_pepper: healthy="Bell_pepper leaf"   diseased=["Bell_pepper leaf spot"]
#   Grape:       healthy="grape leaf"         diseased=["grape leaf black rot"]
#   Tomato:      healthy="Tomato leaf"        diseased=["Tomato Early blight leaf",
#                "Tomato Septoria leaf spot", "Tomato leaf bacterial spot",
#                "Tomato leaf late blight", "Tomato leaf mosaic virus",
#                "Tomato leaf yellow virus", "Tomato mold leaf"]
SPECIALIST = "Tomato"   # one of: Apple, Bell_pepper, Tomato, Grape
HEALTHY_CLASS_FOLDER = "Tomato leaf"           # ← exact folder name for this specialist
DISEASED_CLASS_FOLDERS = ["Tomato Early blight leaf",
                "Tomato Septoria leaf spot", "Tomato leaf bacterial spot",
                "Tomato leaf late blight", "Tomato leaf mosaic virus",
                "Tomato leaf yellow virus", "Tomato mold leaf"]  # ← all diseased classes

# ── Established per-specialist training config — REQUIRED explicit reset ──────
# CONFIG fields NOT explicitly set somewhere in this script silently carry over
# from whatever ran previously in this kernel session (e.g. a prior Grape or
# Classifier run) — this caused a real bug: running Apple with num_classes=2
# and learning_rate=7e-05 leaked in from earlier Grape/Classifier runs. Every
# field below is set explicitly, every run, regardless of specialist, to
# prevent this class of bug entirely rather than relying on remembering to
# check CONFIG state manually each time.
SPECIALIST_CONFIG = {
    "Apple":       {"model": "ConvNeXtLarge", "lr": 1e-4, "num_classes": 3, "img_size": 384},
    "Bell_pepper": {"model": "convnextbase",  "lr": 1e-4, "num_classes": 2, "img_size": 384},
    "Tomato":      {"model": "ConvNeXtLarge", "lr": 1e-4, "num_classes": 8, "img_size": 384},
    "Grape":       {"model": "convnextbase",  "lr": 1e-4, "num_classes": 2, "img_size": 384},
}
_spec_cfg = SPECIALIST_CONFIG[SPECIALIST]
CONFIG["models_to_train"]  = _spec_cfg["model"]
CONFIG["plant_keyword"]    = [SPECIALIST]  # e.g. "Bell_pepper" -> "Bell pepper"
CONFIG["learning_rate"]    = _spec_cfg["lr"]
CONFIG["num_classes"]      = _spec_cfg["num_classes"]
CONFIG["img_size"]         = _spec_cfg["img_size"]

print(f"[Config Reset] SPECIALIST={SPECIALIST} -> "
      f"model={CONFIG['models_to_train']}, plant_keyword={CONFIG['plant_keyword']}, "
      f"lr={CONFIG['learning_rate']}, num_classes={CONFIG['num_classes']}, "
      f"img_size={CONFIG['img_size']}")

ORIGINAL_TRAIN_DIR = f"/home/auc/PlantDoc/Experiments/filtered_{SPECIALIST}/train"
LESION_CSV = "/home/auc/PlantDoc/lesion_coordinates.csv"
COPY_PASTE_ROOT = f"/home/auc/PlantDoc/Datasets/Annotated Images/{SPECIALIST}"

CONDITIONS = ["lesion_unblended", "lesion_blended", "healthy_patch", "random_nonlesion"]
SEEDS = [42, 123, 2024, 779, 1027]   # same seed list used throughout this project

RESULTS_DIR = f"/home/auc/PlantDoc/Experiments/{SPECIALIST}"
os.makedirs(RESULTS_DIR, exist_ok=True)


def build_combined_training_dir(condition, healthy_dir, diseased_dirs, composite_dir, output_dir):
    """
    Builds a training directory combining the ORIGINAL images with this
    condition's generated composites (condition="none" just uses originals,
    unchanged — the control case).
    """
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # rebuild fresh each time — avoid stale-folder issues
    os.makedirs(output_dir)

    # Copy all original class folders unchanged
    for src_dir in [healthy_dir] + diseased_dirs:
        cls_name = os.path.basename(src_dir)
        dst_dir = os.path.join(output_dir, cls_name)
        shutil.copytree(src_dir, dst_dir)

    if condition == "none":
        return  # control condition — originals only, nothing added

    # Add this condition's composites into the HEALTHY class folder (that's
    # what they're based on — a healthy image + a pasted patch, still
    # depicting the same disease as whatever patch was pasted in for the
    # lesion conditions, or remaining a healthy-labeled image structurally
    # for the healthy_patch/random_nonlesion controls — see note below)
    if not os.path.exists(composite_dir):
        print(f"  WARNING: no composites found for {condition} at {composite_dir} — skipping")
        return

    # IMPORTANT LABELING NOTE: lesion_unblended/lesion_blended composites
    # depict a genuine disease patch and should be added to the DISEASED
    # class folder (matching the source lesion's class), not healthy.
    # healthy_patch/random_nonlesion composites are added to the SAME
    # diseased class folder too, for a fair ablation comparison — they are
    # deliberately "fake" positive examples testing whether ANY patch (not
    # necessarily a real lesion) provides the same benefit.
    target_class = os.path.basename(diseased_dirs[0])  # adjust if multi-class disease split needed
    dst_dir = os.path.join(output_dir, target_class)
    for fname in os.listdir(composite_dir):
        shutil.copy(os.path.join(composite_dir, fname), os.path.join(dst_dir, fname))

    print(f"  Added {len(os.listdir(composite_dir))} composite images to {target_class}")


# ── Step 1: generate composites for all conditions (skip "none") ──────────────
print(f"{'='*70}\nSTEP 1: Generating composites for {SPECIALIST}\n{'='*70}")
# NOTE: generate_condition() is used directly here, not imported — it's already
# defined in the previous cell (generate_copy_paste_v2.py's contents), running
# in the same kernel session/namespace.

healthy_dir = os.path.join(ORIGINAL_TRAIN_DIR, HEALTHY_CLASS_FOLDER)
diseased_dir = os.path.join(ORIGINAL_TRAIN_DIR, DISEASED_CLASS_FOLDERS[0])  # primary diseased class for patch sourcing

for condition in CONDITIONS:
    if condition == "none":
        continue
    output_dir = os.path.join(COPY_PASTE_ROOT, condition)
    generate_condition(
        specialist=SPECIALIST, condition=condition,
        lesion_csv_path=LESION_CSV,
        healthy_dir=healthy_dir, diseased_dir=diseased_dir,
        output_dir=output_dir, seed=42,
    )

# ── Step 2: for each condition, build a combined training dir + run multi-seed ─
print(f"\n{'='*70}\nSTEP 2: Multi-seed training per condition\n{'='*70}")

all_results = {}
for condition in CONDITIONS:
    print(f"\n--- Condition: {condition} ---")

    combined_dir = f"./plantdoc_local/working/plantdoc_experiments/filtered_{SPECIALIST}_{condition}/train"
    diseased_dir_paths = [os.path.join(ORIGINAL_TRAIN_DIR, d) for d in DISEASED_CLASS_FOLDERS]
    build_combined_training_dir(
        condition, healthy_dir, diseased_dir_paths,
        os.path.join(COPY_PASTE_ROOT, condition), combined_dir,
    )

    # Point CONFIG at this condition's combined training directory
    CONFIG["dataset_dir_train"] = combined_dir
    CONFIG["experiment_name"] = f"exp_copypaste_v2_{SPECIALIST}_{condition}"
    CONFIG["multi_seed_list"] = SEEDS
    CONFIG["save_model"] = True

    resume_path = os.path.join(RESULTS_DIR, f"{condition}_progress.json")
    results = run_multi_seed_experiment(CONFIG["models_to_train"], resume_path=resume_path)
    all_results[condition] = results

    if results.get("complete"):
        save_multi_seed_results(results, os.path.join(RESULTS_DIR, f"{condition}_FINAL.json"))
        print(f"  {condition}: COMPLETE — {results['summary']['val_accuracy']['mean']*100:.2f}% "
              f"± {results['summary']['val_accuracy']['std']*100:.2f}%")
    else:
        print(f"  {condition}: INCOMPLETE — {len(results['per_seed_results'])}/{len(SEEDS)} seeds done. "
              f"Re-run this script (same resume_path) in a new session to continue.")

print(f"\n{'='*70}\nAll conditions processed. Run analyze_copy_paste_results.py next")
print(f"once every condition shows COMPLETE.\n{'='*70}")

#### ANALYZE COPY-PASTE V2 ABLATION RESULTS  
*(original cell 57)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ANALYZE COPY-PASTE V2 ABLATION RESULTS
# Compares all 5 conditions' multi-seed results, applies the same acceptance
# criteria used throughout this project, and prints a clear summary.
# ═══════════════════════════════════════════════════════════════════════════════

import json
import os
import numpy as np
from scipy import stats

SPECIALISTS = ["Apple", "Bell_pepper", "Tomato", "Grape"]
CONDITIONS = ["none", "lesion_unblended", "lesion_blended", "healthy_patch", "random_nonlesion"]
RESULTS_ROOT = "/home/auc/PlantDoc/Experiments"


def load_condition_result(specialist, condition):
    path = os.path.join(RESULTS_ROOT, specialist, f"{condition}_FINAL.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


print(f"{'='*90}")
print("COPY-PASTE V2 — ABLATION LADDER RESULTS")
print(f"{'='*90}\n")

summary_table = []

for specialist in SPECIALISTS:
    print(f"--- {specialist} ---")
    results_by_condition = {}

    for condition in CONDITIONS:
        result = load_condition_result(specialist, condition)
        if result is None:
            print(f"  {condition:20s}: NOT YET COMPLETE")
            continue
        acc = result["summary"]["val_accuracy"]
        results_by_condition[condition] = acc
        print(f"  {condition:20s}: {acc['mean']*100:6.2f}% \u00b1 {acc['std']*100:5.2f}%  "
              f"(95% CI [{acc['ci_low']*100:.2f}%, {acc['ci_high']*100:.2f}%])")

    if "none" not in results_by_condition:
        print("  (Cannot compute deltas — 'none' control condition not yet complete)\n")
        continue

    baseline = results_by_condition["none"]["mean"]
    print(f"\n  Deltas vs. 'none' baseline ({baseline*100:.2f}%):")
    for condition in CONDITIONS[1:]:
        if condition not in results_by_condition:
            continue
        delta = (results_by_condition[condition]["mean"] - baseline) * 100

        # Check if CIs overlap (rough significance check, consistent with
        # the project's established "does this look like a real effect"
        # approach elsewhere)
        none_ci = results_by_condition["none"]
        cond_ci = results_by_condition[condition]
        overlapping = not (cond_ci["ci_low"] > none_ci["ci_high"] or cond_ci["ci_high"] < none_ci["ci_low"])
        sig_note = "(CIs overlap — not clearly distinguishable)" if overlapping else "(CIs do not overlap — likely real effect)"

        print(f"    {condition:20s}: {delta:+.2f}pp   {sig_note}")
        summary_table.append({
            "specialist": specialist, "condition": condition,
            "delta_pp": delta, "significant": not overlapping,
        })
    print()

# ── Key comparisons that directly test your original hypotheses ───────────────
print(f"{'='*90}")
print("KEY HYPOTHESIS TESTS")
print(f"{'='*90}")
print("""
1. Does copy-paste help AT ALL?
   -> Compare "lesion_blended" vs "none". If positive and CIs don't overlap,
      the technique has genuine value for this specialist.

2. Does blending/matching matter, or is a hard paste just as good?
   -> Compare "lesion_blended" vs "lesion_unblended". If blended wins clearly,
      the feathering/color-matching work was worth it.

3. Does the patch need to be a REAL lesion, or does any patch help?
   -> Compare "lesion_blended" vs "healthy_patch". If lesion_blended wins,
      the technique's value comes from genuine disease content, not just
      "having any patch pasted regardless of content."

4. Does real lesion localization beat the OLD random-crop approach?
   -> Compare "lesion_blended" vs "random_nonlesion" (the old implementation's
      behavior). If lesion_blended wins, the localization redesign was worth
      the annotation effort.
""")

for specialist in SPECIALISTS:
    result = {c: load_condition_result(specialist, c) for c in CONDITIONS}
    if not all(result.values()):
        continue

    def mean(c):
        return result[c]["summary"]["val_accuracy"]["mean"] * 100

    print(f"{specialist}:")
    print(f"  1. copy-paste helps at all:        lesion_blended ({mean('lesion_blended'):.2f}%) vs none ({mean('none'):.2f}%) = {mean('lesion_blended')-mean('none'):+.2f}pp")
    print(f"  2. blending matters:               lesion_blended ({mean('lesion_blended'):.2f}%) vs lesion_unblended ({mean('lesion_unblended'):.2f}%) = {mean('lesion_blended')-mean('lesion_unblended'):+.2f}pp")
    print(f"  3. real lesion content matters:    lesion_blended ({mean('lesion_blended'):.2f}%) vs healthy_patch ({mean('healthy_patch'):.2f}%) = {mean('lesion_blended')-mean('healthy_patch'):+.2f}pp")
    print(f"  4. localization beats old random:  lesion_blended ({mean('lesion_blended'):.2f}%) vs random_nonlesion ({mean('random_nonlesion'):.2f}%) = {mean('lesion_blended')-mean('random_nonlesion'):+.2f}pp")
    print()

# ── Pipeline-wide adoption decision (same ≥4-of-N rule used elsewhere) ────────
n_specialists_improved = sum(
    1 for row in summary_table
    if row["condition"] == "lesion_blended" and row["delta_pp"] > 0 and row["significant"]
)
print(f"{'='*90}")
print(f"ADOPTION DECISION (same criteria as Track 3 augmentation testing)")
print(f"{'='*90}")
print(f"lesion_blended significantly improved {n_specialists_improved} of {len(SPECIALISTS)} specialists tested.")
print("Per the established ≥4-of-N rule: " +
      ("ADOPT as default." if n_specialists_improved >= 4 else
       "Consider per-specialist adoption rather than a pipeline-wide default."))

#### REUSE REAL SAVED WAVE 2 RESULTS AS THE 'none' ABLATION CONDITION  
*(original cell 56)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# REUSE REAL SAVED WAVE 2 RESULTS AS THE 'none' ABLATION CONDITION
# Copies your actual saved multi-seed result files (full per-seed data, all
# 6 metrics — not a reconstructed approximation) into the location
# analyze_copy_paste_results.py expects for the 'none' control condition.
#
# All 4 specialists' values below are confirmed matching the documented
# Wave 2 remediation results exactly:
#   Apple:       94.29% ± 1.96%
#   Bell Pepper: 89.33% ± 5.96%
#   Tomato:      68.06% ± 2.89%
#   Grape:       100.00% ± 0.00%  (note: ci_low/ci_high are NaN in this file —
#                                   harmless, just scipy's t-interval degenerating
#                                   at std=0, not a data error)
# ═══════════════════════════════════════════════════════════════════════════════

import json
import os
import shutil

RESULTS_ROOT = "/home/auc/PlantDoc/Experiments"

# ── EDIT THESE to your actual local file paths ─────────────────────────────────
SOURCE_FILES = {
    "Apple":       "/home/auc/PlantDoc/Apple_results.json",
    "Bell_pepper": "/home/auc/PlantDoc/bell_pepper_results.json",
    "Tomato":      "/home/auc/PlantDoc/Experiments/exp_123_plantDoc_PostDupeRemoval_Tomato/ConvNeXtLarge_multiseed_results.json",
    "Grape":       "/home/auc/PlantDoc/Grape_results.json",
}


def copy_as_none_condition(specialist, source_path):
    if not os.path.exists(source_path):
        print(f"  SKIP {specialist}: source file not found at {source_path}")
        return False

    with open(source_path) as f:
        data = json.load(f)

    n_seeds = len(data.get("per_seed_results", []))
    if n_seeds != 5:
        print(f"  WARNING {specialist}: source file has {n_seeds} seed(s), expected 5 — "
              f"check this is actually the complete, final result before proceeding.")

    out_dir = os.path.join(RESULTS_ROOT, specialist)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "none_FINAL.json")

    data["note"] = ("Copied directly from the completed Wave 2 remediation multi-seed "
                     "result — the 'none' ablation condition is identical to that "
                     "already-completed run (same data, architecture, seeds), so it "
                     "was not re-trained.")

    with open(out_path, "w") as f:
        json.dump(data, f, indent=2)

    acc = data["summary"]["val_accuracy"]
    print(f"  {specialist}: copied real result -> {out_path} "
          f"({acc['mean']*100:.2f}% \u00b1 {acc['std']*100:.2f}%, {n_seeds} seeds)")
    return True


for specialist, path in SOURCE_FILES.items():
    copy_as_none_condition(specialist, path)

print("\nDone. analyze_copy_paste_results.py will now read these directly —")
print("no need to run the 'none' condition through run_ablation_ladder.py.")

## C. Old evaluations (protocol v1: validation = test)

Numbers from these cells are superseded by sections 7–9 of the main notebook.

#### def get_model_save_path_same_run(model_name: str) -> str:  
*(original cell 25)*

In [ ]:
def get_model_save_path_same_run(model_name: str) -> str:
    out = os.path.join(
        CONFIG["output_dir"],
        CONFIG["experiment_name"],
        f"{model_name}{CONFIG['save_format']}"
    )
    os.makedirs(os.path.dirname(out), exist_ok=True)
    return out

def load_model_safe_same_run(model_name: str, fold: int = None) -> tf.keras.Model: 
    suffix = f"_fold{fold}" if fold is not None else ""
    path   = get_model_save_path_same_run(f"{model_name}{suffix}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}\nRun training first.")
    print(f"[Ensemble] Loading {model_name} from {path}")
    return tf.keras.models.load_model(path)

model_mob = load_model_safe_same_run("efficientnetb0")
_, val_ds_mob, class_names_ens = get_datasets("efficientnetb0")

all_preds, all_labels = [], []
for images, labels in val_ds_mob:
    preds = model_mob.predict(images, verbose=0)
    all_preds.append(np.argmax(preds, axis=1))
    all_labels.append(np.argmax(labels.numpy(), axis=1))

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
acc = np.mean(y_pred == y_true)

print(f"Val Accuracy : {acc:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")

model_mob = load_model_safe_same_run("EfficientNetV2S")
_, val_ds_mob, class_names_ens = get_datasets("EfficientNetV2S")

all_preds, all_labels = [], []
for images, labels in val_ds_mob:
    preds = model_mob.predict(images, verbose=0)
    all_preds.append(np.argmax(preds, axis=1))
    all_labels.append(np.argmax(labels.numpy(), axis=1))

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
acc = np.mean(y_pred == y_true)

print(f"Val Accuracy : {acc:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")

#### for model_name, (model, history) in trained_models.items():  
*(original cell 27)*

In [ ]:
for model_name, (model, history) in trained_models.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{model_name} — {CONFIG['experiment_name']}", fontsize=13)

    axes[0].plot(history.history["accuracy"],     label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["loss"],     label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"],
                             f"{model_name}_history.png")
    os.makedirs(os.path.dirname(plot_path), exist_ok=True)
    plt.savefig(plot_path, dpi=120)
    plt.show()
    print(f"Saved to {plot_path}")

#### def get_predictions(model, ds) -> np.ndarray:  
*(original cell 29)*

In [ ]:
def get_predictions(model, ds) -> np.ndarray:
    all_probs = []
    for batch_images, _ in ds:
        probs = model.predict(batch_images, verbose=0)
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


def get_true_labels(ds, num_classes: int) -> np.ndarray:
    all_labels = []
    for _, batch_labels in ds:
        all_labels.append(batch_labels.numpy())
    return np.argmax(np.concatenate(all_labels, axis=0), axis=1)


def soft_vote(probs_list: list, weights: list) -> np.ndarray:
    assert abs(sum(weights) - 1.0) < 1e-5, "Ensemble weights must sum to 1.0"
    blended = sum(w * p for w, p in zip(weights, probs_list))
    return np.argmax(blended, axis=1)


def plot_confusion_matrix(y_true, y_pred, class_names):
    cm  = confusion_matrix(y_true, y_pred)
    sz  = max(10, len(class_names))
    fig, ax = plt.subplots(figsize=(sz, sz - 2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {CONFIG['experiment_name']}")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    cm_path = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"],
                           "confusion_matrix.png")
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f"Confusion matrix saved to {cm_path}")


# ── Run ensemble ──────────────────────────────────────────────────────────────
weights = CONFIG["ensemble_weights"]

_, val_ds_mob, class_names_ens = get_datasets("efficientnetb0")
_, val_ds_eff, _               = get_datasets("EfficientNetV2S")

model_mob = load_model_safe_same_run("efficientnetb0")
model_eff = load_model_safe_same_run("EfficientNetV2S")

print("\n[Ensemble] Running inference...")
probs_mob = get_predictions(model_mob, val_ds_mob)
probs_eff = get_predictions(model_eff, val_ds_eff)

y_true = get_true_labels(val_ds_mob, CONFIG["num_classes"])

acc_mob = np.mean(np.argmax(probs_mob, axis=1) == y_true)
acc_eff = np.mean(np.argmax(probs_eff, axis=1) == y_true)
y_pred  = soft_vote([probs_mob, probs_eff], weights)
acc_ens = np.mean(y_pred == y_true)

print(f"\n========== Ensemble Summary ==========")
print(f"  Experiment  : {CONFIG['experiment_name']}")
print(f"  MobileNetV2 : {acc_mob:.4f}")
print(f"  EffNetB0    : {acc_eff:.4f}")
print(f"  Ensemble    : {acc_ens:.4f}  (weights {weights})")
print(f"======================================")

print("\n[Ensemble] Per-class report:")
print(classification_report(y_true, y_pred, target_names=class_names_ens, digits=3))

plot_confusion_matrix(y_true, y_pred, class_names_ens)

#### STANDALONE ENSEMBLE CELL  
*(original cell 30)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STANDALONE ENSEMBLE CELL
# Run this cell independently — no other cells need to be run first except
# the imports cell (Cell 2).
#
# Configure the 4 variables below, then run.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Configure these ───────────────────────────────────────────────────────────
MODEL_A_NAME   = "convnextbase"    # used for preprocessing fn selection
MODEL_A_PATH   = MODEL_PATHS["tomato_ensemble_convnextbase"]

MODEL_B_NAME   = "convnextlarge"   # used for preprocessing fn selection
MODEL_B_PATH   = MODEL_PATHS["tomato_ensemble_convnextlarge"]

ENSEMBLE_WEIGHTS = [.15, .85]        # must sum to 1.0  [Model A, Model B]

DATASET_TEST_DIR = "./plantdoc_local/working/plantdoc_experiments/filtered_Tomato/test"
IMG_SIZE         = 384
BATCH_SIZE       = 32
NUM_CLASSES      = 27
OUTPUT_DIR       = "./plantdoc_local/working/plantdoc_experiments/standalone_ensemble"
# ─────────────────────────────────────────────────────────────────────────────

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import mobilenet_v2, efficientnet, efficientnet_v2

os.makedirs(OUTPUT_DIR, exist_ok=True)

def _standalone_preprocess_fn(model_name: str):
    name = model_name.lower()
    if "mobilenet" in name:
        return mobilenet_v2.preprocess_input
    elif "v2s" in name or "efficientnetv2" in name:
        return efficientnet_v2.preprocess_input
    else:
        return efficientnet.preprocess_input   # covers b0, b1, b2, b3, b4

def _load_val_ds(model_name: str):
    preprocess_fn = _standalone_preprocess_fn(model_name)

    raw_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_TEST_DIR,
        seed=42,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=None,
        label_mode="int",
        shuffle=False,
    )
    class_names = raw_ds.class_names
    actual_classes = len(class_names)
    if actual_classes != NUM_CLASSES:
        print(f"  WARNING: NUM_CLASSES={NUM_CLASSES} but found {actual_classes}. Using {actual_classes}.")

    def process(image, label):
        image = preprocess_fn(image)
        label = tf.one_hot(label, actual_classes)
        return image, label

    ds = raw_ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, class_names

def _get_predictions(model, ds):
    all_probs = []
    for batch_images, _ in ds:
        probs = model.predict(batch_images, verbose=0)
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

def _get_true_labels(ds):
    all_labels = []
    for _, batch_labels in ds:
        all_labels.append(batch_labels.numpy())
    return np.argmax(np.concatenate(all_labels, axis=0), axis=1)

def _plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm  = confusion_matrix(y_true, y_pred)
    sz  = max(10, len(class_names))
    fig, ax = plt.subplots(figsize=(sz, sz - 2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix\n{os.path.basename(MODEL_A_PATH)} + {os.path.basename(MODEL_B_PATH)}")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved to {save_path}")

# ── Load models ───────────────────────────────────────────────────────────────
print(f"Loading Model A: {MODEL_A_PATH}")
if not os.path.exists(MODEL_A_PATH):
    raise FileNotFoundError(f"Model A not found: {MODEL_A_PATH}")
model_a = tf.keras.models.load_model(MODEL_A_PATH)

print(f"Loading Model B: {MODEL_B_PATH}")
if not os.path.exists(MODEL_B_PATH):
    raise FileNotFoundError(f"Model B not found: {MODEL_B_PATH}")
model_b = tf.keras.models.load_model(MODEL_B_PATH)

# ── Load datasets (separate preprocessing per model) ─────────────────────────
print("\nLoading datasets...")
val_ds_a, class_names = _load_val_ds(MODEL_A_NAME)
val_ds_b, _           = _load_val_ds(MODEL_B_NAME)

# ── Run inference ─────────────────────────────────────────────────────────────
print(f"\nRunning inference — {MODEL_A_NAME}...")
probs_a = _get_predictions(model_a, val_ds_a)

print(f"Running inference — {MODEL_B_NAME}...")
probs_b = _get_predictions(model_b, val_ds_b)

y_true = _get_true_labels(val_ds_a)

# ── Individual accuracies ─────────────────────────────────────────────────────
acc_a = np.mean(np.argmax(probs_a, axis=1) == y_true)
acc_b = np.mean(np.argmax(probs_b, axis=1) == y_true)

# ── Soft vote ─────────────────────────────────────────────────────────────────
assert abs(sum(ENSEMBLE_WEIGHTS) - 1.0) < 1e-5, "ENSEMBLE_WEIGHTS must sum to 1.0"
blended = ENSEMBLE_WEIGHTS[0] * probs_a + ENSEMBLE_WEIGHTS[1] * probs_b
y_pred  = np.argmax(blended, axis=1)
acc_ens = np.mean(y_pred == y_true)

# ── Results ───────────────────────────────────────────────────────────────────
print(f"\n========== Standalone Ensemble Summary ==========")
print(f"  Model A ({MODEL_A_NAME}): {acc_a:.4f}")
print(f"  Model B ({MODEL_B_NAME}): {acc_b:.4f}")
print(f"  Ensemble (weights {ENSEMBLE_WEIGHTS}): {acc_ens:.4f}")
print(f"=================================================")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

_plot_confusion_matrix(y_true, y_pred, class_names,
                       os.path.join(OUTPUT_DIR, "confusion_matrix.png"))

# ── Save results to txt ───────────────────────────────────────────────────────
results_path = os.path.join(OUTPUT_DIR, "ensemble_results.txt")
with open(results_path, "w") as f:
    f.write(f"Model A: {MODEL_A_PATH}\n")
    f.write(f"Model B: {MODEL_B_PATH}\n")
    f.write(f"Weights: {ENSEMBLE_WEIGHTS}\n\n")
    f.write(f"Model A accuracy : {acc_a:.4f}\n")
    f.write(f"Model B accuracy : {acc_b:.4f}\n")
    f.write(f"Ensemble accuracy: {acc_ens:.4f}\n\n")
    f.write(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print(f"Results saved to {results_path}")

#### F1-WEIGHTED ENSEMBLE (PROPER NORMALIZED VERSION) — ConvNeXtBase + ConvNeXtLarge  
*(original cell 31)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# F1-WEIGHTED ENSEMBLE (PROPER NORMALIZED VERSION) — ConvNeXtBase + ConvNeXtLarge
# Tomato Specialist
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report

# ── Configure here ────────────────────────────────────────────────────────────
MODEL_A_NAME = "convnextbase"
MODEL_A_PATH = MODEL_PATHS["tomato_ensemble_convnextbase"]   # ← update

MODEL_B_NAME = "convnextlarge"
MODEL_B_PATH = MODEL_PATHS["tomato_ensemble_convnextlarge"]  # ← update

DATASET_TEST_DIR    = "./plantdoc_local/working/plantdoc_experiments/filtered_Tomato/test"  # filtered tomato test dir
IMG_SIZE    = 384
BATCH_SIZE  = 32

# ── Load models ───────────────────────────────────────────────────────────────
print(f"Loading Model A: {MODEL_A_PATH}")
model_a = tf.keras.models.load_model(MODEL_A_PATH)

print(f"Loading Model B: {MODEL_B_PATH}")
model_b = tf.keras.models.load_model(MODEL_B_PATH)

# ── Load datasets (same preprocessing assumed, both ConvNeXt family) ──────────
val_ds_a, class_names = _load_val_ds(MODEL_A_NAME)
val_ds_b, _           = _load_val_ds(MODEL_B_NAME)

# ── Get predictions ───────────────────────────────────────────────────────────
probs_a = _get_predictions(model_a, val_ds_a)
probs_b = _get_predictions(model_b, val_ds_b)
y_true  = _get_true_labels(val_ds_a)

y_pred_a = np.argmax(probs_a, axis=1)
y_pred_b = np.argmax(probs_b, axis=1)

acc_a = np.mean(y_pred_a == y_true)
acc_b = np.mean(y_pred_b == y_true)
print(f"\n{MODEL_A_NAME} accuracy: {acc_a:.4f}")
print(f"{MODEL_B_NAME} accuracy: {acc_b:.4f}")

# ── Calculate per-class F1 scores ──────────────────────────────────────────────
f1_a = f1_score(y_true, y_pred_a, average=None)
f1_b = f1_score(y_true, y_pred_b, average=None)

print(f"\n{MODEL_A_NAME} F1 per class:")
for cn, f1 in zip(class_names, f1_a):
    print(f"  {cn}: {f1:.3f}")

print(f"\n{MODEL_B_NAME} F1 per class:")
for cn, f1 in zip(class_names, f1_b):
    print(f"  {cn}: {f1:.3f}")

# ── Proper normalized weighted ensemble (same structure as cassava version) ───
def true_hard_routing(probs_a, probs_b, f1_a, f1_b, y_pred_a, y_pred_b):
    """
    For each sample, if Model A's predicted class has higher F1 for Model A
    than Model B's predicted class has for Model B, trust Model A's full prediction.
    Otherwise trust Model B's full prediction.
    """
    final_preds = []
    for i in range(len(probs_a)):
        pred_a_class = y_pred_a[i]
        pred_b_class = y_pred_b[i]
        
        f1_for_a_pred = f1_a[pred_a_class]
        f1_for_b_pred = f1_b[pred_b_class]
        
        if f1_for_a_pred >= f1_for_b_pred:
            final_preds.append(pred_a_class)
        else:
            final_preds.append(pred_b_class)
    
    return np.array(final_preds)

y_pred_hard2 = true_hard_routing(probs_a, probs_b, f1_a, f1_b, y_pred_a, y_pred_b)
acc_hard2 = np.mean(y_pred_hard2 == y_true)
print(f"True hard routing accuracy: {acc_hard2:.4f}")

print(f"\n========== F1-Weighted Ensemble Results ==========")
print(f"  {MODEL_A_NAME}: {acc_a:.4f}")
print(f"  {MODEL_B_NAME}: {acc_b:.4f}")
print(f"  Ensemble:      {acc_hard2:.4f}")
print(f"====================================================")

print("\nEnsemble per-class report:")
print(classification_report(y_true, y_pred_hard2, target_names=class_names, digits=3))

# Find a sample where models disagree
disagree_idx = np.where(y_pred_a != y_pred_b)[0]
print(f"Number of disagreements: {len(disagree_idx)}")

# ── Save results ───────────────────────────────────────────────────────────────
results_path = os.path.join(CONFIG["output_dir"], "tomato_f1_ensemble_results.txt")
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w") as f:
    f.write(f"{MODEL_A_NAME} accuracy: {acc_a:.4f}\n")
    f.write(f"{MODEL_B_NAME} accuracy: {acc_b:.4f}\n")
    f.write(f"Ensemble accuracy: {acc_hard2:.4f}\n\n")
    f.write(f"{MODEL_A_NAME} F1 per class: {dict(zip(class_names, f1_a.round(3)))}\n")
    f.write(f"{MODEL_B_NAME} F1 per class: {dict(zip(class_names, f1_b.round(3)))}\n\n")
    f.write(classification_report(y_true, y_pred_hard2, target_names=class_names, digits=3))
print(f"\nResults saved to {results_path}")

#### ── Single model test evaluation  
*(original cell 33)*

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# ── Single model test evaluation ──────────────────────────────────────────────
MODEL_NAME = "noisy"
MODEL_PATH = MODEL_PATHS["apple_copypaste_test_model"]

DATASET_TEST_DIR = "./plantdoc_local/working/plantdoc_experiments/filtered_Apple/test"
IMG_SIZE         = 384
BATCH_SIZE       = 32
NUM_CLASSES      = 3
OUTPUT_DIR       = "./plantdoc_local/working/plantdoc_experiments/single_model_eval"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Loading: {MODEL_PATH}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Not found: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)

val_ds, class_names = _load_val_ds(MODEL_NAME)

probs  = _get_predictions(model, val_ds)
y_true = _get_true_labels(val_ds)
y_pred = np.argmax(probs, axis=1)
acc    = np.mean(y_pred == y_true)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
print(f"\n========== Single Model Results ==========")
print(f"  Model     : {MODEL_NAME}")
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"==========================================")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

_plot_confusion_matrix(y_true, y_pred, class_names,
                       os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_confusion_matrix.png"))

# ── Identify misclassified images ─────────────────────────────────────────────
misclassified_indices = np.where(y_pred != y_true)[0]
print(f"\nTotal misclassified: {len(misclassified_indices)}\n")

raw_files = []
for class_name in class_names:
    class_dir = os.path.join(DATASET_TEST_DIR, class_name)
    for fname in sorted(os.listdir(class_dir)):
        raw_files.append(os.path.join(class_dir, fname))

for idx in misclassified_indices:
    confidence = probs[idx][y_pred[idx]]
    print(f"File: {raw_files[idx]}")
    print(f"  True: {class_names[y_true[idx]]} | Predicted: {class_names[y_pred[idx]]} | Confidence: {confidence:.3f}\n")

#### PLANTVILLAGE → PLANTDOC ZERO-SHOT GENERALIZATION EVALUATION  
*(original cell 34)*

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report
import os
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════════
# PLANTVILLAGE → PLANTDOC ZERO-SHOT GENERALIZATION EVALUATION
# Handles class-space mismatch: PlantVillage model may have classes with no
# PlantDoc equivalent (e.g. Apple Black Rot). These are tracked separately as
# "invalid" predictions rather than silently excluded, since a real deployment
# would still be exposed to this failure mode.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Configure here per specialist ──────────────────────────────────────────────
MODEL_NAME       = "noisy"
MODEL_PATH       = MODEL_PATHS["plantvillage_potato_model"]
DATASET_TEST_DIR = "./plantdoc_local/working/plantdoc_experiments/filtered_Potato/test"   # PlantDoc test set
IMG_SIZE         = 384
BATCH_SIZE       = 32
OUTPUT_DIR       = "./plantdoc_local/working/plantdoc_experiments/plantvillage_generalization_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map EVERY class the PlantVillage model can predict to its PlantDoc equivalent.
# Use None for any PlantVillage class that has NO corresponding PlantDoc class
# (e.g. Apple Black Rot). These become "invalid" predictions when they occur.
#
# Example for Apple:

INVALID_LABEL = "__NO_PLANTDOC_EQUIVALENT__"
CLASS_MAP = {
    "Potato___Early_blight"           :         "Potato leaf early blight",
    "Potato___Late_blight"                  :         "Potato leaf late blight",
    "Potato___healthy"                     :          INVALID_LABEL
    # "Peach___healthy"                            :         "Peach leaf",
    # "Raspberry___healthy"                        :         "Raspberry leaf",
    # "Soybean___healthy"                          :         "Soyabean leaf",
    # "Squash___Powdery_mildew"                    :          None,
    # "Strawberry___Leaf_scorch"                   :         "Squash Powdery mildew leaf",
    # "Strawberry___healthy"                       :         "Strawberry leaf",
    # "Tomato___healthy"                               :         "Tomato leaf",
}
# CLASS_MAP = {
#     # ← fill in for the current specialist being tested
# }



# ── Load model ──────────────────────────────────────────────────────────────────
print(f"Loading: {MODEL_PATH}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Not found: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)

# ── Load PlantDoc test set ──────────────────────────────────────────────────────
val_ds, plantdoc_class_names = _load_val_ds(MODEL_NAME)   # PlantDoc ground-truth classes, sorted alphabetically
probs = _get_predictions(model, val_ds)                    # shape (N, num_plantvillage_classes)
y_true_idx = _get_true_labels(val_ds)                       # indices into plantdoc_class_names

# ── Determine PlantVillage class order (must match model's output order) ──────
# This should be the exact class list/order the PlantVillage model was trained
# and saved with. If you have it stored elsewhere, load it here instead.
plantvillage_class_names = list(CLASS_MAP.keys())
if len(plantvillage_class_names) != probs.shape[1]:
    raise ValueError(
        f"CLASS_MAP has {len(plantvillage_class_names)} entries but model outputs "
        f"{probs.shape[1]} classes. Update CLASS_MAP to match the model's actual "
        f"training class order exactly."
        )

# ── Map every PlantVillage prediction to its PlantDoc equivalent (or invalid) ──
y_pred_pv_idx = np.argmax(probs, axis=1)
y_pred_mapped_names = [
    CLASS_MAP[plantvillage_class_names[i]] if CLASS_MAP[plantvillage_class_names[i]] is not None
    else INVALID_LABEL
    for i in y_pred_pv_idx
]

y_true_names = [plantdoc_class_names[i] for i in y_true_idx]

# ── Build unified label space for scoring (PlantDoc classes + invalid marker) ──
scoring_labels = sorted(set(plantdoc_class_names) | {INVALID_LABEL})
label_to_idx = {name: i for i, name in enumerate(scoring_labels)}

y_true_scored = np.array([label_to_idx[name] for name in y_true_names])
y_pred_scored = np.array([label_to_idx[name] for name in y_pred_mapped_names])

acc = np.mean(y_pred_scored == y_true_scored)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_scored, y_pred_scored, average="weighted", zero_division=0
)

# ── Invalid-prediction rate (spurious "no-equivalent-class" firing) ────────────
n_invalid = sum(1 for name in y_pred_mapped_names if name == INVALID_LABEL)
pct_invalid = 100 * n_invalid / len(y_pred_mapped_names)

print(f"\n========== PlantVillage → PlantDoc Generalization: {MODEL_NAME} ==========")
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  -------------------------------------------------------")
print(f"  Invalid predictions (no PlantDoc equivalent): {n_invalid} / {len(y_pred_mapped_names)} ({pct_invalid:.2f}%)")
print(f"===========================================================")

print("\nPer-class report (includes invalid-prediction class if it occurred):")
print(classification_report(
    y_true_scored, y_pred_scored,
    labels=list(range(len(scoring_labels))),   # ← add this
    target_names=scoring_labels, digits=3, zero_division=0
))

_plot_confusion_matrix(
    y_true_scored, y_pred_scored, scoring_labels,
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_generalization_confusion_matrix.png")
)

# ── Identify misclassified images (including those misfired to invalid class) ─
misclassified_indices = np.where(y_pred_scored != y_true_scored)[0]
print(f"\nTotal misclassified: {len(misclassified_indices)}\n")

raw_files = []
for class_name in plantdoc_class_names:
    class_dir = os.path.join(DATASET_TEST_DIR, class_name)
    for fname in sorted(os.listdir(class_dir)):
        raw_files.append(os.path.join(class_dir, fname))

for idx in misclassified_indices:
    confidence = probs[idx][y_pred_pv_idx[idx]]
    true_name = scoring_labels[y_true_scored[idx]]
    pred_name = scoring_labels[y_pred_scored[idx]]
    fpath = raw_files[idx] if idx < len(raw_files) else f"(index {idx})"
    print(f"File: {fpath}")
    print(f"  True: {true_name} | Predicted: {pred_name} | Confidence: {confidence:.3f}\n")

# ── Save summary ────────────────────────────────────────────────────────────────
results_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_generalization_results.txt")
with open(results_path, "w") as f:
    f.write(f"Model: {MODEL_PATH}\n")
    f.write(f"PlantDoc test set: {DATASET_TEST_DIR}\n\n")
    f.write(f"Accuracy  : {acc:.4f}\n")
    f.write(f"Precision : {precision:.4f}\n")
    f.write(f"Recall    : {recall:.4f}\n")
    f.write(f"F1 Score  : {f1:.4f}\n\n")
    f.write(f"Invalid predictions (no PlantDoc equivalent): {n_invalid} / {len(y_pred_mapped_names)} ({pct_invalid:.2f}%)\n\n")
    f.write(classification_report(
        y_true_scored, y_pred_scored,
        target_names=scoring_labels, digits=3, zero_division=0
    ))
print(f"\nResults saved to {results_path}")

#### FILTER-BANK TTA EVALUATION — Confidence Boost Analysis  
*(original cell 36)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FILTER-BANK TTA EVALUATION — Confidence Boost Analysis
# Measures: among baseline-correct predictions, what % gained confidence after TTA
#
# Filter bank updated to remove smoothing-based filters (Bilateral Denoise,
# redundant Strong CLAHE since CLAHE is already applied during preprocessing).
# Replaced with sharpening/edge and color-difference filters that preserve
# fine spatial detail — important since diseases often manifest as small,
# localized lesions that smoothing filters can erase.
# ═══════════════════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import tensorflow as tf
import os
from sklearn.metrics import classification_report, precision_recall_fscore_support

# ── Configure here ────────────────────────────────────────────────────────────
MODEL_NAME = "ConvNeXtLarge_classifier"          # used to select correct preprocess_fn
MODEL_PATH = MODEL_PATHS["classifier_tta_eval_model"]   # ← update per specialist
TEST_DIR   = "./plantdoc_local/working/crop_classifier_v3/test"        # ← update per specialist
IMG_SIZE   = 384
ORIGINAL_WEIGHT = 1.0   # set to whichever weight you found best in the sweep
OUTPUT_DIR = "./plantdoc_local/working/plantdoc_experiments/filter_tta_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILTER_NAMES = ["Original", "Sharpened", "Laplacian Edge Enhance", "Saturation Boost", "Color-Diff Boost (LAB a/b)"]

# ── Filter bank ────────────────────────────────────────────────────────────────
def apply_filter_bank(image):
    """
    image: numpy array, RGB, uint8, shape (H, W, 3)
    Returns a list of filtered versions including the original.
    Order matches FILTER_NAMES exactly. Index 0 must always be the original.

    All filters here either sharpen/enhance edges or boost color differences.
    No smoothing/blurring/denoising filters are included, since these were
    found to erase fine-grained lesion texture (e.g. Bilateral Denoise hurt
    Corn Gray Leaf Spot and Potato specifically). "Strong CLAHE" was also
    removed since input images already have standard CLAHE applied during
    preprocessing, making a second CLAHE pass redundant/compounding rather
    than a genuinely new view of the image.
    """
    filtered_versions = []

    # 1. Original (baseline)
    filtered_versions.append(image)

    # 2. Sharpened (unsharp mask) — enhances lesion edges via blend with blurred version
    gaussian = cv2.GaussianBlur(image, (0, 0), 3)
    sharpened = cv2.addWeighted(image, 1.5, gaussian, -0.5, 0)
    filtered_versions.append(sharpened)

    # 3. Laplacian edge enhancement — different sharpening mechanism than unsharp mask;
    #    detects edges directly via second derivative, adds them back without any
    #    blurring step involved
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = np.uint8(np.absolute(laplacian))
    laplacian_3ch = cv2.cvtColor(laplacian, cv2.COLOR_GRAY2RGB)
    laplacian_enhanced = cv2.addWeighted(image, 1.0, laplacian_3ch, 0.3, 0)
    filtered_versions.append(laplacian_enhanced)

    # 4. Saturation boost — emphasizes color-based symptoms (HSV color space)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.4, 0, 255)
    saturated = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    filtered_versions.append(saturated)

    # 5. Color-difference boost (LAB a/b channels) — amplifies color deviation
    #    direction (green-red, blue-yellow) without touching brightness or
    #    spatial sharpness; complements Saturation Boost with a perceptually
    #    different color space, potentially more precise for disease-specific
    #    discoloration (e.g. grayish vs. brownish lesion tones)
    lab_cd = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab_cd[:, :, 1] = np.clip((lab_cd[:, :, 1] - 128) * 1.5 + 128, 0, 255)  # 'a' channel
    lab_cd[:, :, 2] = np.clip((lab_cd[:, :, 2] - 128) * 1.5 + 128, 0, 255)  # 'b' channel
    color_diff_boosted = cv2.cvtColor(lab_cd.astype(np.uint8), cv2.COLOR_LAB2RGB)
    filtered_versions.append(color_diff_boosted)

    return filtered_versions


# ── Preprocessing selector (reuses your existing pipeline convention) ─────────
def _make_preprocess_fn(model_name: str):
    name = model_name.lower()
    if "mobilenet" in name:
        return tf.keras.applications.mobilenet_v2.preprocess_input
    elif "v2s" in name or "v2l" in name or "v2m" in name or "efficientnetv2" or "noisy" in name:
        return tf.keras.applications.efficientnet_v2.preprocess_input
    elif "convnext" in name:
        return tf.keras.applications.convnext.preprocess_input
    elif "noisy" in name or "efficientnetb" in name:
        return tf.keras.applications.efficientnet.preprocess_input
    else:
        return lambda x: x / 255.0


preprocess_fn = _make_preprocess_fn(MODEL_NAME)


def get_original_and_tta_probs(model, image_uint8, original_weight=1.0):
    """
    Returns (original_probs, tta_weighted_probs) for a single image.
    original_probs: the baseline, no-TTA prediction (just the unfiltered image)
    tta_weighted_probs: the weighted average across the full filter bank
    """
    filtered_versions = apply_filter_bank(image_uint8)

    preds = []
    weights = []
    for idx, filt_img in enumerate(filtered_versions):
        img_float = tf.cast(filt_img, tf.float32)
        img_float = preprocess_fn(img_float)
        img_float = tf.expand_dims(img_float, axis=0)
        pred = model(img_float, training=False).numpy()[0]
        preds.append(pred)
        weights.append(original_weight if idx == 0 else 1.0)

    original_probs = preds[0]  # index 0 is always the unfiltered original
    preds = np.array(preds)
    weights = np.array(weights)
    tta_weighted_probs = np.average(preds, axis=0, weights=weights)

    return original_probs, tta_weighted_probs


def evaluate_confidence_boost(model, test_dir, img_size, original_weight=1.0):
    raw_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir, seed=42,
        image_size=(img_size, img_size),
        batch_size=None, label_mode="int", shuffle=False,
    )
    class_names = raw_ds.class_names

    all_original_probs, all_tta_probs, all_labels = [], [], []
    total = sum(1 for _ in raw_ds)

    for i, (image, label) in enumerate(raw_ds):
        print(f"  Processing {i+1}/{total}", end="\r")
        image_uint8 = tf.cast(image, tf.uint8).numpy()

        orig_probs, tta_probs = get_original_and_tta_probs(
            model, image_uint8, original_weight=original_weight
        )
        all_original_probs.append(orig_probs)
        all_tta_probs.append(tta_probs)
        all_labels.append(label.numpy())

    all_original_probs = np.array(all_original_probs)
    all_tta_probs = np.array(all_tta_probs)
    y_true = np.array(all_labels)

    y_pred_original = np.argmax(all_original_probs, axis=1)
    y_pred_tta = np.argmax(all_tta_probs, axis=1)

    acc_original = np.mean(y_pred_original == y_true)
    acc_tta = np.mean(y_pred_tta == y_true)

    # ── Confidence-boost analysis on baseline-CORRECT predictions only ────────
    baseline_correct_mask = (y_pred_original == y_true)
    n_baseline_correct = baseline_correct_mask.sum()

    # Confidence = probability assigned to the TRUE class (not just argmax,
    # since we're only looking at cases that were already correct, so the
    # true class IS the argmax class for these)
    true_class_idx = y_true[baseline_correct_mask]
    row_idx = np.arange(len(y_true))[baseline_correct_mask]

    conf_original_on_correct = all_original_probs[row_idx, true_class_idx]
    conf_tta_on_correct      = all_tta_probs[row_idx, true_class_idx]

    confidence_delta = conf_tta_on_correct - conf_original_on_correct
    n_boosted   = (confidence_delta > 0).sum()
    n_decreased = (confidence_delta < 0).sum()
    n_unchanged = (confidence_delta == 0).sum()

    pct_boosted   = 100 * n_boosted / n_baseline_correct
    pct_decreased = 100 * n_decreased / n_baseline_correct
    pct_unchanged = 100 * n_unchanged / n_baseline_correct

    mean_conf_original = conf_original_on_correct.mean()
    mean_conf_tta      = conf_tta_on_correct.mean()
    mean_delta         = confidence_delta.mean()

    # ── Did TTA flip any baseline-correct predictions to WRONG? ──────────────
    still_correct_mask = (y_pred_tta[row_idx] == true_class_idx)
    n_flipped_to_wrong = (~still_correct_mask).sum()
    pct_flipped_to_wrong = 100 * n_flipped_to_wrong / n_baseline_correct

    print(f"\n\n========== Confidence Boost Analysis ==========")
    print(f"  Model                    : {MODEL_NAME}")
    print(f"  Filter bank              : {FILTER_NAMES}")
    print(f"  Baseline accuracy        : {acc_original:.4f}")
    print(f"  TTA accuracy             : {acc_tta:.4f}")
    print(f"  ------------------------------------------------")
    print(f"  Baseline-correct images  : {n_baseline_correct}")
    print(f"  Confidence boosted       : {n_boosted} ({pct_boosted:.2f}%)")
    print(f"  Confidence decreased     : {n_decreased} ({pct_decreased:.2f}%)")
    print(f"  Confidence unchanged     : {n_unchanged} ({pct_unchanged:.2f}%)")
    print(f"  ------------------------------------------------")
    print(f"  Mean confidence (before) : {mean_conf_original:.4f}")
    print(f"  Mean confidence (after)  : {mean_conf_tta:.4f}")
    print(f"  Mean confidence delta    : {mean_delta:+.4f}")
    print(f"  ------------------------------------------------")
    print(f"  Flipped to WRONG by TTA  : {n_flipped_to_wrong} ({pct_flipped_to_wrong:.2f}%)")
    print(f"=================================================")

    return {
        "acc_original": acc_original,
        "acc_tta": acc_tta,
        "n_baseline_correct": n_baseline_correct,
        "n_boosted": n_boosted,
        "pct_boosted": pct_boosted,
        "n_decreased": n_decreased,
        "pct_decreased": pct_decreased,
        "n_unchanged": n_unchanged,
        "pct_unchanged": pct_unchanged,
        "mean_conf_original": mean_conf_original,
        "mean_conf_tta": mean_conf_tta,
        "mean_delta": mean_delta,
        "n_flipped_to_wrong": n_flipped_to_wrong,
        "pct_flipped_to_wrong": pct_flipped_to_wrong,
        "y_pred_tta": y_pred_tta,
        "y_true": y_true,
        "class_names": class_names,
    }


# ── Run ────────────────────────────────────────────────────────────────────────
print(f"Loading: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)

results = evaluate_confidence_boost(model, TEST_DIR, IMG_SIZE, original_weight=ORIGINAL_WEIGHT)

print("\nFull TTA per-class report:")
print(classification_report(
    results["y_true"], results["y_pred_tta"],
    target_names=results["class_names"], digits=3
))

# ── Save results ───────────────────────────────────────────────────────────────
results_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_confidence_boost_results.txt")
with open(results_path, "w") as f:
    f.write(f"Model: {MODEL_PATH}\n")
    f.write(f"Filter bank: {FILTER_NAMES}\n\n")
    f.write(f"Baseline accuracy        : {results['acc_original']:.4f}\n")
    f.write(f"TTA accuracy             : {results['acc_tta']:.4f}\n\n")
    f.write(f"Baseline-correct images  : {results['n_baseline_correct']}\n")
    f.write(f"Confidence boosted       : {results['n_boosted']} ({results['pct_boosted']:.2f}%)\n")
    f.write(f"Confidence decreased     : {results['n_decreased']} ({results['pct_decreased']:.2f}%)\n")
    f.write(f"Confidence unchanged     : {results['n_unchanged']} ({results['pct_unchanged']:.2f}%)\n\n")
    f.write(f"Mean confidence (before) : {results['mean_conf_original']:.4f}\n")
    f.write(f"Mean confidence (after)  : {results['mean_conf_tta']:.4f}\n")
    f.write(f"Mean confidence delta    : {results['mean_delta']:+.4f}\n\n")
    f.write(f"Flipped to WRONG by TTA  : {results['n_flipped_to_wrong']} ({results['pct_flipped_to_wrong']:.2f}%)\n\n")
    f.write(classification_report(
        results["y_true"], results["y_pred_tta"],
        target_names=results["class_names"], digits=3
    ))
print(f"\nResults saved to {results_path}")

#### ── TTA Evaluation  
*(original cell 37)*

In [ ]:
# ── TTA Evaluation ────────────────────────────────────────────────────────────
MODEL_NAME = "convnextbase"
MODEL_PATH = MODEL_PATHS["apple_tta_eval_model"]
TEST_DIR   = "./plantdoc_local/working/plantdoc_experiments/filtered_Apple/test"

N_AUGMENTS = 5

print(f"Loading: {MODEL_PATH}")
model_tta = tf.keras.models.load_model(MODEL_PATH)

preprocess_fn = _make_preprocess_fn(MODEL_NAME)
img_size = 384

aug_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

raw_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, seed=42,
    image_size=(img_size, img_size),
    batch_size=None, label_mode="int", shuffle=False,
)
class_names = raw_ds.class_names
num_classes  = len(class_names)

all_preds, all_labels = [], []
total = sum(1 for _ in raw_ds)

for i, (image, label) in enumerate(raw_ds):
    print(f"  Processing {i+1}/{total}", end="\r")
    preds = []
    for _ in range(N_AUGMENTS):
        aug_img = aug_layer(image, training=True)
        aug_img = preprocess_fn(aug_img)
        aug_img = tf.expand_dims(aug_img, axis=0)
        preds.append(model_tta(aug_img, training=False).numpy()[0])
    all_preds.append(np.mean(preds, axis=0))
    all_labels.append(label.numpy())

y_pred = np.argmax(all_preds, axis=1)
y_true = np.array(all_labels)
acc    = np.mean(y_pred == y_true)

from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print(f"\n========== TTA Results ({N_AUGMENTS} augments) ==========")
print(f"  Model     : {MODEL_NAME}")
print(f"  Plant     : {CONFIG.get('plant_keyword', 'All')}")
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"==============================================")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm_path = os.path.join(
    CONFIG["output_dir"], CONFIG["experiment_name"],
    f"{MODEL_NAME}_{CONFIG.get('plant_keyword','all')}_tta_confusion_matrix.png"
)
os.makedirs(os.path.dirname(cm_path), exist_ok=True)
_plot_confusion_matrix(y_true, y_pred, class_names, cm_path)

## D. One-off scans & utilities

#### PlantVillage Grape model on PlantDoc black-rot images (zero-shot check)  
*(original cell 35)*

In [ ]:
import os
import numpy as np
import tensorflow as tf

# ── Configure ────────────────────────────────────────────────────────────────
MODEL_PATH = MODEL_PATHS["plantvillage_grape_model"]
BLACK_ROT_DIR = "./plantdoc_local/working/plantdoc_experiments/filtered_grape/test/grape leaf black rot"  # PlantDoc black rot folder
IMG_SIZE = 384

# Must match the model's actual output order exactly
PLANTVILLAGE_CLASSES = [
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
]

# ── Load model ──────────────────────────────────────────────────────────────
model = tf.keras.models.load_model(MODEL_PATH)

# ── Inspect every Black Rot test image's raw prediction ──────────────────────
for fname in sorted(os.listdir(BLACK_ROT_DIR)):
    fpath = os.path.join(BLACK_ROT_DIR, fname)
    img = tf.keras.utils.load_img(fpath, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = tf.keras.utils.img_to_array(img)
    img_arr = tf.expand_dims(img_arr, axis=0)

    # NOTE: apply whichever preprocess_input your model actually expects here
    # (this must match _make_preprocess_fn's logic for this model)
    img_arr = tf.keras.applications.convnext.preprocess_input(img_arr)

    pred = model(img_arr, training=False).numpy()[0]

    print(f"{fname}:")
    for cls, p in zip(PLANTVILLAGE_CLASSES, pred):
        print(f"    {cls:45s}: {p:.3f}")
    top_class = PLANTVILLAGE_CLASSES[np.argmax(pred)]
    print(f"  -> Predicted: {top_class} ({np.max(pred):.3f})\n")

#### Aspect-ratio scan of the raw dataset (flags extreme shapes)  
*(original cell 38)*

In [ ]:
import os
from PIL import Image

# ── Configure here ────────────────────────────────────────────────────────────
SCAN_DIR = "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset"   # root folder to scan (train and/or test)
MAX_ASPECT_RATIO = 2   # flag if width/height OR height/width exceeds this
OUTPUT_REPORT = "./plantdoc_local/working/aspect_ratio_flags.txt"

# ── Scan all images recursively ────────────────────────────────────────────────
flagged = []
total_checked = 0

for root, dirs, files in os.walk(SCAN_DIR):
    for fname in files:
        if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            continue

        fpath = os.path.join(root, fname)
        total_checked += 1

        try:
            with Image.open(fpath) as img:
                w, h = img.size
        except Exception as e:
            print(f"  ERROR reading {fpath}: {e}")
            continue

        ratio_wh = w / h
        ratio_hw = h / w
        max_ratio = max(ratio_wh, ratio_hw)

        if max_ratio > MAX_ASPECT_RATIO:
            flagged.append({
                "path": fpath,
                "width": w,
                "height": h,
                "ratio": round(max_ratio, 2),
            })

        if total_checked % 200 == 0:
            print(f"  Checked {total_checked} images so far... ({len(flagged)} flagged)", end="\r")

print(f"\n\nTotal images checked: {total_checked}")
print(f"Total flagged (aspect ratio > {MAX_ASPECT_RATIO}:1): {len(flagged)}")

# ── Print and save flagged images, sorted worst-first ─────────────────────────
flagged.sort(key=lambda x: x["ratio"], reverse=True)

print(f"\n{'Ratio':>7} | {'Width':>6} | {'Height':>6} | Path")
for f in flagged:
    print(f"{f['ratio']:>7.2f} | {f['width']:>6} | {f['height']:>6} | {f['path']}")

with open(OUTPUT_REPORT, "w") as f_out:
    f_out.write(f"Total images checked: {total_checked}\n")
    f_out.write(f"Flagged (aspect ratio > {MAX_ASPECT_RATIO}:1): {len(flagged)}\n\n")
    f_out.write(f"{'Ratio':>7} | {'Width':>6} | {'Height':>6} | Path\n")
    for f in flagged:
        f_out.write(f"{f['ratio']:>7.2f} | {f['width']:>6} | {f['height']:>6} | {f['path']}\n")

print(f"\nReport saved to {OUTPUT_REPORT}")

#### Aspect-ratio resize check (visual before/after)  
*(original cell 41)*

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# ── Configure here ────────────────────────────────────────────────────────────
IMG_SIZE = 384

# Paste the paths you want to inspect (e.g. copy the worst offenders from your report)
IMAGES_TO_CHECK = [
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Potato leaf late blight/Potato leaf late blight (22).jpg",
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Corn Gray leaf spot/Corn Gray leaf spot (31).jpg",
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Corn rust leaf/Corn rust leaf (74).jpg",
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Potato leaf late blight/Potato leaf late blight (73).jpg",
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Corn rust leaf/Corn rust leaf (97).jpg",
    "./plantdoc_local/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train/Corn Gray leaf spot/Corn Gray leaf spot (29).jpg",
]

# ── Plot original vs resized side by side ──────────────────────────────────────
n = len(IMAGES_TO_CHECK)
fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))

for i, path in enumerate(IMAGES_TO_CHECK):
    img = Image.open(path).convert("RGB")
    w, h = img.size

    resized = img.resize((IMG_SIZE, IMG_SIZE))  # same as tf's image_dataset_from_directory default

    ax_orig = axes[i, 0] if n > 1 else axes[0]
    ax_resized = axes[i, 1] if n > 1 else axes[1]

    ax_orig.imshow(img)
    ax_orig.set_title(f"Original ({w}x{h})\n{os.path.basename(path)}", fontsize=9)
    ax_orig.axis("off")

    ax_resized.imshow(resized)
    ax_resized.set_title(f"Resized to {IMG_SIZE}x{IMG_SIZE}", fontsize=9)
    ax_resized.axis("off")

plt.tight_layout()
plt.savefig("./plantdoc_local/working/aspect_ratio_resize_check.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved comparison to ./plantdoc_local/working/aspect_ratio_resize_check.png")

#### Collect multi-seed results folders into an Excel summary  
*(original cell 42)*

In [ ]:
import os
import json
import re
import pandas as pd

# ── Configure here ────────────────────────────────────────────────────────────
ROOT_DIR   = "./plantdoc_local/working/plantdoc_experiments"
OUTPUT_XLSX = "./plantdoc_local/working/multiseed_results_summary.xlsx"

# ── Find all folders containing "Multi" in their name ──────────────────────────
multi_folders = [
    d for d in os.listdir(ROOT_DIR)
    if os.path.isdir(os.path.join(ROOT_DIR, d)) and "multi" in d.lower()
]
print(f"Found {len(multi_folders)} folder(s) containing 'Multi':")
for d in multi_folders:
    print(f"  {d}")

# ── Parse folder name: augmentation type = segment right before "MultiSeed",
#    specialist = segment after the LAST underscore ─────────────────────────────
def parse_folder_name(folder_name):
    parts = folder_name.split("_")

    # Specialist = last segment
    specialist = parts[-1]

    # Augmentation type = segment immediately before whichever part contains "multiseed"
    aug_type = None
    for i, p in enumerate(parts):
        if "multiseed" in p.lower():
            if i > 0:
                aug_type = parts[i - 1]
            break

    if aug_type is None:
        aug_type = "UNKNOWN"

    return aug_type, specialist


# ── Collect rows from every .json file in every matching folder ───────────────
rows = []

for folder in multi_folders:
    folder_path = os.path.join(ROOT_DIR, folder)
    aug_type, specialist = parse_folder_name(folder)

    json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    if not json_files:
        print(f"  WARNING: no .json file found in {folder} — skipping")
        continue

    for json_file in json_files:
        json_path = os.path.join(folder_path, json_file)
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
        except Exception as e:
            print(f"  ERROR reading {json_path}: {e}")
            continue

        model_name = data.get("model_name", "")
        seeds = data.get("seeds", [])
        summary = data.get("summary", {})

        row = {
            "Folder": folder,
            "Augmentation": aug_type,
            "Specialist": specialist,
            "Model": model_name,
            "Num Seeds": len(seeds),
            "Seeds": ", ".join(str(s) for s in seeds),
        }

        # Add mean ± std (combined string) and CI (combined string) per metric
        for metric_name, stats in summary.items():
            mean = stats.get("mean")
            std = stats.get("std")
            ci_low = stats.get("ci_low")
            ci_high = stats.get("ci_high")

            mean_pct = mean * 100 if mean is not None else None
            std_pct = std * 100 if std is not None else None
            ci_low_pct = ci_low * 100 if ci_low is not None else None
            ci_high_pct = ci_high * 100 if ci_high is not None else None

            row[f"{metric_name}"] = (
                f"{mean_pct:.2f} \u00b1 {std_pct:.2f}"
                if mean_pct is not None and std_pct is not None else ""
            )
            row[f"{metric_name}_CI"] = (
                f"[{ci_low_pct:.2f}, {ci_high_pct:.2f}]"
                if ci_low_pct is not None and ci_high_pct is not None else ""
            )

        rows.append(row)
        print(f"  Parsed: {folder} -> Augmentation={aug_type}, Specialist={specialist}")

# ── Build DataFrame and save to Excel ───────────────────────────────────────────
if not rows:
    print("\nNo results collected — check ROOT_DIR and folder naming.")
else:
    df = pd.DataFrame(rows)

    # Order columns: identifying info first, then metrics grouped together
    id_cols = ["Folder", "Augmentation", "Specialist", "Model", "Num Seeds", "Seeds"]
    metric_cols = [c for c in df.columns if c not in id_cols]
    df = df[id_cols + sorted(metric_cols)]

    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"\nSaved {len(rows)} row(s) to {OUTPUT_XLSX}")

#### Collect experiment config.txt files into a CSV  
*(original cell 39)*

In [ ]:
import os
import re
import csv

# ── Configure here ────────────────────────────────────────────────────────────
CONFIGS_DIR = "./plantdoc_local/working/all_configs_collected"
OUTPUT_CSV  = "./plantdoc_local/working/configs_summary.csv"

PARAMETERS = [
    "experiment_name",
    "models_to_train",
    "img_size",
    "num_classes",
    "epochs",
    "learning_rate",
    "fine_tune",
    "fine_tune_lr",
    "fine_tune_at_epoch",
    "use_cyclical_lr",
    "use_augmentation",
    "use_cutmix",
    "use_mixup",
    "use_clahe",
    "use_focal_loss",
    "random_seed",
    "plant_keyword",
    "batch_size",
]

def parse_config_file(filepath):
    """Parse a config.txt file and extract parameter values as a dict."""
    values = {}
    with open(filepath, "r") as f:
        content = f.read()

    for param in PARAMETERS:
        pattern = rf"^\s*{re.escape(param)}\s*:\s*(.+)$"
        match = re.search(pattern, content, re.MULTILINE)
        if match:
            values[param] = match.group(1).strip()
        else:
            values[param] = "-"
    return values

# ── Process all config files ────────────────────────────────────────────────────
config_files = sorted([f for f in os.listdir(CONFIGS_DIR) if f.endswith(".txt")])

rows = []
for fname in config_files:
    filepath = os.path.join(CONFIGS_DIR, fname)
    parsed = parse_config_file(filepath)
    rows.append(parsed)
    print(f"Parsed: {fname}")

# ── Write to CSV ─────────────────────────────────────────────────────────────────
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=PARAMETERS)
    writer.writeheader()
    writer.writerows(rows)

print(f"\nTotal configs processed: {len(rows)}")
print(f"CSV saved to: {OUTPUT_CSV}")

# ── Provide download link ───────────────────────────────────────────────────────
from IPython.display import FileLink
display(FileLink(OUTPUT_CSV))

#### Collect best-epoch metrics of all experiments into a CSV  
*(original cell 40)*

In [ ]:
import os
import csv
import pandas as pd

# ── Configure here ────────────────────────────────────────────────────────────
SOURCE_DIR  = "./plantdoc_local/working/plantdoc_experiments"
OUTPUT_CSV  = "./plantdoc_local/working/best_metrics_summary.csv"

# ── Process each experiment folder ─────────────────────────────────────────────
rows = []

experiment_folders = sorted(os.listdir(SOURCE_DIR))

for folder_name in experiment_folders:
    folder_path = os.path.join(SOURCE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    # Find the single CSV file in this subfolder
    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    if len(csv_files) == 0:
        print(f"  Skipped (no CSV): {folder_name}")
        continue
    if len(csv_files) > 1:
        print(f"  WARNING: multiple CSVs in {folder_name}, using first: {csv_files[0]}")

    csv_path = os.path.join(folder_path, csv_files[0])

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  ERROR reading {csv_path}: {e}")
        continue

    # Confirm required columns exist
    required_cols = ["accuracy", "loss", "val_accuracy", "val_loss"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  WARNING: {folder_name} missing columns {missing} — skipping")
        continue

    # Find the row with the best (max) val_accuracy
    best_idx = df["val_accuracy"].idxmax()
    best_row = df.loc[best_idx]

    rows.append({
        "experiment_name":    folder_name,
        "best_epoch":         best_idx,
        "best_val_accuracy":  round(best_row["val_accuracy"], 4),
        "val_loss_at_best":   round(best_row["val_loss"], 4),
        "train_accuracy_at_best": round(best_row["accuracy"], 4),
        "train_loss_at_best":     round(best_row["loss"], 4),
        "total_epochs_logged":    len(df),
        "final_val_accuracy":      round(df["val_accuracy"].iloc[-1], 4),  # last epoch, for spike detection
        "overfit_gap_at_best":    round(best_row["accuracy"] - best_row["val_accuracy"], 4),
    })

    print(f"Processed: {folder_name} -> best val_accuracy={best_row['val_accuracy']:.4f} at epoch {best_idx}")

# ── Write summary CSV ────────────────────────────────────────────────────────────
output_df = pd.DataFrame(rows)
output_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nTotal experiments processed: {len(rows)}")
print(f"Summary saved to: {OUTPUT_CSV}")

from IPython.display import FileLink
display(FileLink(OUTPUT_CSV))

#### RECOVER SEED 1027's METRICS FROM ITS SURVIVING CHECKPOINT  
*(original cell 48)*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RECOVER SEED 1027's METRICS FROM ITS SURVIVING CHECKPOINT
# Crashed just before the final epoch's validation — but ModelCheckpoint saves
# the best epoch continuously DURING training (whenever a new best val_accuracy
# is hit), not just at the very end, so the surviving .keras file should still
# reflect a genuine best-epoch result, not a broken/partial one.
# ═══════════════════════════════════════════════════════════════════════════════

import tensorflow as tf
import gc

# ── Configure here ────────────────────────────────────────────────────────────
CRASHED_SEED = 1027
CHECKPOINT_PATH = os.path.join(
    CONFIG["output_dir"], CONFIG["experiment_name"],
    f"{CONFIG['models_to_train']}_seed{CRASHED_SEED}{CONFIG['save_format']}"
)

# ── Reload and evaluate ─────────────────────────────────────────────────────────
print(f"Loading checkpoint for seed {CRASHED_SEED}: {CHECKPOINT_PATH}")
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at {CHECKPOINT_PATH} — check CONFIG['experiment_name'] "
        f"and CONFIG['models_to_train'] match what was actually running when it crashed."
    )

model = tf.keras.models.load_model(CHECKPOINT_PATH)

_, val_ds, class_names = get_datasets(CONFIG["models_to_train"])
metrics = _compute_run_metrics(model, val_ds, class_names)   # reuses your existing function

print(f"\n{'='*55}")
print(f"RECOVERED METRICS — seed {CRASHED_SEED}")
print(f"{'='*55}")
print(f"  val_accuracy:       {metrics['val_accuracy']:.4f}")
print(f"  precision:          {metrics['precision']:.4f}")
print(f"  recall:             {metrics['recall']:.4f}")
print(f"  f1:                 {metrics['f1']:.4f}")
print(f"  macro_f1:           {metrics['macro_f1']:.4f}")
print(f"  balanced_accuracy:  {metrics['balanced_accuracy']:.4f}")
print(f"\nRecord these alongside your other 4 seeds' manually-recorded metrics.")

# ── Clean up ─────────────────────────────────────────────────────────────────
del model
tf.keras.backend.clear_session()
gc.collect()